#### Setup

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import logging
import time

from project import Dataset, Route
from project.Metaheuristics import Metaheuristic
from project import settings as cfg

logging.basicConfig(
    level=logging.INFO,
    format="%(message)s",
)
logger = logging.getLogger("TCC")

logging.getLogger("MESA.mesa.model").setLevel(logging.WARNING)

BASE_PATH = Path("..")


# Experiment controls
N_RUNS = cfg.N_RUNS
SEED_START = cfg.SEED_START
PARALLEL_AGENTS = cfg.PARALLEL_AGENTS

# Time-budget controls
TOTAL_MAIN_BUDGET_SEC = cfg.TOTAL_MAIN_BUDGET_SEC

# MAS controls
MAS_POOL_SIZE = cfg.MAS_POOL_SIZE
MAS_SEED_POLICY = cfg.MAS_SEED_POLICY
MAS_N_STEPS = cfg.MAS_N_STEPS

# Tunable parameters
SA_PARAMS = cfg.SA_PARAMS
TS_PARAMS = cfg.TS_PARAMS
GA_PARAMS = cfg.GA_PARAMS
MAS_SA_PARAMS = cfg.MAS_SA_PARAMS
MAS_TS_PARAMS = cfg.MAS_TS_PARAMS
MAS_GA_PARAMS = cfg.MAS_GA_PARAMS

# RL controls
MAS_RL_PARAMS = cfg.MAS_RL_PARAMS

# Comparison controls
D_GAPS = cfg.D_GAPS
K_GAPS = cfg.K_GAPS
TTT_TARGETS = cfg.TTT_TARGETS
METHOD_LABELS_ORIGINAL = ["TS", "SA", "GA", "MAS", "MAS RL"]
METHOD_LABELS_ADDITIONAL = [
    "MAS SA",
    "MAS TS",
    "MAS GA",
    "MAS (TS+SA)",
    "MAS RL (TS+SA)",
]
METHOD_LABELS = METHOD_LABELS_ORIGINAL + METHOD_LABELS_ADDITIONAL
METHOD_COUNT = len(METHOD_LABELS)
INSTANCE_GROUPS = cfg.INSTANCE_GROUPS

# External tuner defaults
LAST_TUNING_REPORT = cfg.LAST_TUNING_REPORT

# Calculated configs
INSTANCE_COUNT = cfg.INSTANCE_COUNT
TARGET_METHOD_RUNS = cfg.TARGET_METHOD_RUNS
TARGET_METHOD_RUN_SEC = cfg.TARGET_METHOD_RUN_SEC

#???
ACTIVE_BATCH_MAX_INSTANCES_PER_GROUP = None
ACTIVE_BATCH_EXECUTE = True
ACTIVE_BATCH_VERBOSE = False

def flatten_hybrid_instances(groups: dict[str, list[str]]) -> list[str]:
    '''Flattens the instance groups into a single list of instance names, preserving a preferred order.'''
    preferred_order = ["C1", "C2", "R1", "R2", "RC1", "RC2"]
    instance_names: list[str] = []
    for group_name in preferred_order:
        instance_names.extend(groups.get(group_name, []))

    for group_name, values in groups.items():
        if group_name not in preferred_order:
            instance_names.extend(values)

    deduplicated = list(dict.fromkeys(instance_names))
    return deduplicated

def build_reference_table(
    instance_names: list[str],
) -> dict[str, dict[str, object]]:
    '''
    Builds a reference table for the given instance names. 
    The reference table maps instance names to their reference solution values
    (number of vehicles and total distance).
    '''
    reference_table: dict[str, dict[str, object]] = {}

    for instance_name in instance_names:
        dataset_path = BASE_PATH / "Datasets" / instance_name
        solution_path = BASE_PATH / "Solutions" / instance_name
        if (not dataset_path.exists()) or (not solution_path.exists()):
            continue

        ds_ref = Dataset(dataset_path)
        route_ref = Route(dataset=ds_ref)
        route_ref.read_solution(solution_path)
        k_ref, d_ref = route_ref.cost_function()

        reference_table[instance_name] = {
            "instance": instance_name,
            "dataset_name": ds_ref.name,
            "vehicles_ref": int(k_ref),
            "distance_ref": float(d_ref),
        }

    return reference_table

HYBRID_INSTANCE_LIST = flatten_hybrid_instances(INSTANCE_GROUPS)
HYBRID_REFERENCE_TABLE = build_reference_table(HYBRID_INSTANCE_LIST)

logger.info(f"--- Comparison setup ---")
logger.info("")
logger.info(f"Runs = {N_RUNS}")
logger.info(f"Methods = {METHOD_COUNT} ({', '.join(METHOD_LABELS)})")
logger.info(f"Instances = {INSTANCE_COUNT}")
logger.info(f"Target time budget = {TOTAL_MAIN_BUDGET_SEC:.2f}s")
logger.info(f"Target method runs = {TARGET_METHOD_RUNS:.2f}")
logger.info(f"Target time per method run = {TARGET_METHOD_RUN_SEC:.2f}s")
logger.info(f"MAS steps = {MAS_N_STEPS}")
logger.info("")
logger.info(f"Active SA params = {SA_PARAMS}")
logger.info(f"Active TS params = {TS_PARAMS}")
logger.info(f"Active GA params = {GA_PARAMS}")
logger.info("")
logger.info(f"Active MAS SA params = {MAS_SA_PARAMS}")
logger.info(f"Active MAS TS params = {MAS_TS_PARAMS}")
logger.info(f"Active MAS GA params = {MAS_GA_PARAMS}")
logger.info(f"Active MAS RL params = {MAS_RL_PARAMS}")


# Exploring Solomon Benchmark

## Comprehending diferences between datasets

In [ ]:
# type hints for better readability and IDE support
axes: np.ndarray
ax: plt.Axes

paths = [
    BASE_PATH / "Datasets" / "C101.txt",
    BASE_PATH / "Datasets" / "R101.txt",
    BASE_PATH / "Datasets" / "RC101.txt",
    BASE_PATH / "Datasets" / "RC201.txt",
]

datasets = [Dataset(p) for p in paths]

avg_window_times = [
    (ds.customers_df['due_date'] - ds.customers_df['ready_time']).mean()
    for ds in datasets
 ]

for ds, avg_time in zip(datasets, avg_window_times):
    logger.info(f"{ds.name} - Average Time Window: {avg_time:.2f}")

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()

for ax, ds in zip(axes, datasets):
    ax.scatter(ds.customers_df.x[1:], ds.customers_df.y[1:], c='blue', label='Customers', alpha=0.6, s=10)
    ax.scatter(ds.customers_df.x[0], ds.customers_df.y[0], c='red', marker='s', s=60, label='Depot')
    ax.set_title(ds.name)
    ax.set_xlabel("X Coordinate")
    ax.set_ylabel("Y Coordinate")
    ax.set_xlim(0, 100)
    ax.set_ylim(0, 100)
    ax.grid(True, linestyle='--', alpha=0.5)

for ax in axes[len(datasets):]:
    ax.axis("off")

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper right")
fig.suptitle("Solomon Instances - Geographic Distribution")
fig.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()

## Testing base code

Validations for cost function, vizualization methods

In [ ]:
# Read solution and plot routes with cost
solution_path = BASE_PATH / "Solutions" / "rc201.txt"
dataset_path = BASE_PATH / "Datasets" / "rc201.txt"

ds = Dataset(dataset_path)
route = Route(dataset=ds)
route.read_solution(solution_path)

feasibility = route.is_feasible()
total_cost = route.calc_total_distance()
num_subroutes = route.calc_total_vehicles()
total_wait = route.calc_total_waiting_time()
total_service = route.calc_total_service_time()
total_time = route.calc_total_time()

logger.info(f"Feasibility of the solution: {'Feasible' if feasibility else 'Infeasible'}")
logger.info(f"Number of routes in solution: {num_subroutes}")
logger.info(f"Total distance (cost): {total_cost:.2f}")

if (round(total_cost, 2) == 1406.94) and (num_subroutes == 4) and feasibility:
    logger.info("The total cost matches the expected values from literature.")
    # NOTE: This is a validation check to ensure the solution's cost is as expected
    # Check value from literature here: https://www.sintef.no/projectweb/top/vrptw/100-customers/

route.plot_routes('Benchmark Solution - RC201')

# Metaheuristics 

Creating a runner to encapsulate multiple running on metaheuristic algorithms.

In [ ]:
class MetaheuristicRunner:
    def __init__(self, algo_cls: type, dataset: Dataset, runs: int = N_RUNS, seed_start: int = 42, **algo_kwargs: object) -> None:
        self.algo_cls = algo_cls
        self.dataset = dataset
        self.runs = runs
        self.seed_start = seed_start
        self.algo_kwargs = dict(algo_kwargs)

    def run(self, plot_best: bool = False, best_plot_title: str | None = None) -> dict[str, object]:
        runtimes: list[float] = []
        vehicles: list[int] = []
        distances: list[float] = []
        total_iterations: list[int] = []

        best_route: Route | None = None
        best_k: int | None = None
        best_d: float | None = None
        best_history_t: list[float] = []
        best_history_k: list[int] = []
        best_history_d: list[float] = []

        run_records: list[dict[str, object]] = []

        logger.info(
            f"{self.algo_cls.__name__} setup | runs={self.runs}, "
            f"seed_start={self.seed_start}, params={self.algo_kwargs}"
        )

        for run_idx in range(self.runs):
            seed = self.seed_start + run_idx
            params = dict(self.algo_kwargs)
            params["seed"] = seed

            logger.info(
                f"{self.algo_cls.__name__} run {run_idx + 1}/{self.runs} | seed={seed}"
            )

            algo: Metaheuristic = self.algo_cls(dataset=self.dataset, **params)

            start_time = time.process_time()
            route = algo.solve()
            elapsed = float(time.process_time() - start_time)

            k_run, d_run = route.cost_function()

            history_trace = [
                {
                    "elapsed_sec": float(t),
                    "vehicles": int(k),
                    "distance": float(d),
                }
                for t, k, d in algo.history_trace()
            ]
            if (not history_trace) or history_trace[-1]["elapsed_sec"] < elapsed:
                history_trace.append(
                    {
                        "elapsed_sec": elapsed,
                        "vehicles": int(k_run),
                        "distance": float(d_run),
                    }
                )

            run_iterations = int(len(history_trace))

            runtimes.append(elapsed)
            vehicles.append(int(k_run))
            distances.append(float(d_run))
            total_iterations.append(run_iterations)

            run_record = {
                "run": int(run_idx + 1),
                "seed": int(seed),
                "runtime_sec": float(elapsed),
                "vehicles": int(k_run),
                "distance": float(d_run),
                "iterations": int(run_iterations),
                "trace": history_trace,
            }
            run_records.append(run_record)

            logger.info(
                f"Run {run_idx + 1} result | runtime: {elapsed:.4f}s | "
                f"K: {int(k_run)}, D: {float(d_run):.2f} | iters: {run_iterations}"
            )

            if best_route is None or best_k is None or best_d is None or (k_run, d_run) < (best_k, best_d):
                best_route = route
                best_k, best_d = int(k_run), float(d_run)
                best_history_t = [float(point["elapsed_sec"]) for point in history_trace]
                best_history_k = [int(point["vehicles"]) for point in history_trace]
                best_history_d = [float(point["distance"]) for point in history_trace]

                logger.info(
                    f"Run {run_idx + 1} status | new best so far -> K: {best_k}, D: {best_d:.2f}"
                )
            else:
                logger.info(
                    f"Run {run_idx + 1} status | best so far unchanged -> K: {best_k}, D: {best_d:.2f}"
                )

        dataset_key = f"{self.dataset.name.lower()}.txt"
        summary = {
            "method_label": self.algo_cls.__name__,
            "dataset_key": dataset_key,
            "runs": self.runs,
            "avg_runtime_sec": float(np.mean(runtimes)) if runtimes else float("nan"),
            "std_runtime_sec": float(np.std(runtimes, ddof=1)) if len(runtimes) > 1 else 0.0,
            "avg_vehicles": float(np.mean(vehicles)) if vehicles else float("nan"),
            "std_vehicles": float(np.std(vehicles, ddof=1)) if len(vehicles) > 1 else 0.0,
            "avg_distance": float(np.mean(distances)) if distances else float("nan"),
            "std_distance": float(np.std(distances, ddof=1)) if len(distances) > 1 else 0.0,
            "avg_total_iterations": float(np.mean(total_iterations)) if total_iterations else 0.0,
            "std_total_iterations": float(np.std(total_iterations, ddof=1)) if len(total_iterations) > 1 else 0.0,
            "best_vehicles": int(best_k) if best_k is not None else None,
            "best_distance": float(best_d) if best_d is not None else None,
            "best_route": best_route,
            "best_history_t": best_history_t,
            "best_history_k": best_history_k,
            "best_history_d": best_history_d,
            "run_records": run_records,
        }

        if summary["best_vehicles"] is None or summary["best_distance"] is None:
            logger.info(
                f"{self.algo_cls.__name__} over {self.runs} runs | no feasible run produced a best route"
            )
        else:
            logger.info(
                f"{self.algo_cls.__name__} over {self.runs} runs | "
                f"avg runtime: {summary['avg_runtime_sec']:.4f}s ± {summary['std_runtime_sec']:.4f} | "
                f"avg K: {summary['avg_vehicles']:.2f} ± {summary['std_vehicles']:.2f} | "
                f"avg D: {summary['avg_distance']:.2f} ± {summary['std_distance']:.2f} | "
                f"avg iters: {summary['avg_total_iterations']:.2f} ± {summary['std_total_iterations']:.2f} | "
                f"best: (K={summary['best_vehicles']}, D={summary['best_distance']:.2f})"
            )

        if plot_best and best_route is not None:
            title = best_plot_title or f"{self.algo_cls.__name__} Best of {self.runs} Runs"
            best_route.plot_routes(title)

            if best_history_k and best_history_d:
                n_iterations = len(best_history_k)
                x = range(1, n_iterations + 1)

                fig, ax1 = plt.subplots(figsize=(6, 3))
                ax1.plot(x, best_history_k, color='tab:blue', label='Vehicles')
                ax1.set_xlabel("Iteration")
                ax1.set_ylabel("Vehicles (K)", color='tab:blue')
                ax1.tick_params(axis='y', labelcolor='tab:blue')

                ax2 = ax1.twinx()
                ax2.plot(x, best_history_d, color='tab:orange', label='Distance')
                ax2.set_ylabel("Distance (D)", color='tab:orange')
                ax2.tick_params(axis='y', labelcolor='tab:orange')

                plt.title(f"{self.algo_cls.__name__} Best Run Cost History")
                fig.tight_layout()
                plt.show()

        return summary

# Hybrid Evaluation (Budget + Time-to-Target)

In [ ]:
def first_hit_time(
    trace: list[dict[str, float | int]],
    *,
    k_target: int | None = None,
    d_target: float | None = None,
) -> float:
    for point in trace:
        k_ok = True if k_target is None else int(point["vehicles"]) <= int(k_target)
        d_ok = True if d_target is None else float(point["distance"]) <= float(d_target)
        if k_ok and d_ok:
            return float(point["elapsed_sec"])
    return float("nan")


def best_pair_within_budget(
    trace: list[dict[str, float | int]],
    budget_sec: float,
) -> tuple[int, float] | None:
    best_pair: tuple[int, float] | None = None
    for point in trace:
        if float(point["elapsed_sec"]) > float(budget_sec):
            continue
        pair = (int(point["vehicles"]), float(point["distance"]))
        if best_pair is None or pair < best_pair:
            best_pair = pair
    return best_pair


def finite_values(values: list[float]) -> list[float]:
    return [float(value) for value in values if np.isfinite(value)]


def median_or_nan(values: list[float]) -> float:
    finite = finite_values(values)
    return float(np.median(finite)) if finite else float("nan")


def method_categories(values: pd.Series) -> list[str]:
    observed = [str(v) for v in pd.unique(values.astype(str))]
    ordered = [m for m in METHOD_LABELS if m in observed]
    tail = [m for m in observed if m not in ordered]
    return ordered + tail


def sort_by_method(df: pd.DataFrame, method_col: str, extra_sort_cols: list[str] | None = None) -> pd.DataFrame:
    if df.empty:
        return df
    result = df.copy()
    categories = method_categories(result[method_col])
    result["_method_order"] = pd.Categorical(
        result[method_col].astype(str),
        categories=categories,
        ordered=True,
    )
    sort_cols = list(extra_sort_cols or []) + ["_method_order"]
    result = result.sort_values(sort_cols).drop(columns=["_method_order"]).reset_index(drop=True)
    return result


def plot_ttt_ecdf(
    ttt_times_by_method: dict[str, list[float]],
    title: str,
    xlabel: str,
) -> None:
    fig, ax = plt.subplots(figsize=(7, 4))
    plotted = False
    for method_label in METHOD_LABELS:
        if method_label not in ttt_times_by_method:
            continue
        times = ttt_times_by_method[method_label]
        finite = np.array(sorted(finite_values(times)), dtype=float)
        if finite.size == 0:
            continue
        y = np.arange(1, finite.size + 1, dtype=float) / float(len(times))
        ax.step(finite, y, where="post", label=method_label)
        plotted = True

    for method_label, times in ttt_times_by_method.items():
        if method_label in METHOD_LABELS:
            continue
        finite = np.array(sorted(finite_values(times)), dtype=float)
        if finite.size == 0:
            continue
        y = np.arange(1, finite.size + 1, dtype=float) / float(len(times))
        ax.step(finite, y, where="post", label=method_label)
        plotted = True

    if not plotted:
        logger.info(f"No successful runs available for plot: {title}")
        plt.close(fig)
        return

    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel("Success Probability")
    ax.set_ylim(0.0, 1.02)
    ax.grid(True, linestyle="--", alpha=0.4)
    ax.legend()
    fig.tight_layout()
    plt.show()


summary_var_by_label = {
    "TS": "ts_summary",
    "SA": "sa_summary",
    "GA": "ga_summary",
    "MAS": "mas_summary",
    "MAS RL": "mas_rl_summary",
    "MAS SA": "mas_sa_summary",
    "MAS TS": "mas_ts_summary",
    "MAS GA": "mas_ga_summary",
    "MAS (TS+SA)": "mas_tssa_summary",
    "MAS RL (TS+SA)": "mas_rl_tssa_summary",
}
method_eval_order = [m for m in METHOD_LABELS if m in summary_var_by_label]
method_eval_order += [
    m for m in METHOD_LABELS
    if m in summary_var_by_label and m not in method_eval_order
]

method_summaries: dict[str, dict[str, object]] = {}
for method_label in method_eval_order:
    summary = globals().get(summary_var_by_label[method_label], None)
    if isinstance(summary, dict) and summary.get("run_records"):
        method_summaries[method_label] = summary

if "ds" in globals() and ds is not None:
    current_instance_key = f"{ds.name.lower()}.txt"
else:
    current_instance_key = None

reference = None if current_instance_key is None else HYBRID_REFERENCE_TABLE.get(current_instance_key, None)

if not method_summaries:
    logger.info(
        "Hybrid evaluation skipped: no method summaries found. Run TS / SA / GA / MAS / MAS RL / MAS SA / MAS TS / MAS GA / MAS (TS+SA) / MAS RL (TS+SA) sections first."
    )
elif reference is None:
    logger.info(
        f"Hybrid evaluation skipped: no benchmark reference available for instance '{current_instance_key}'."
    )
else:
    k_ref = int(reference["vehicles_ref"])
    d_ref = float(reference["distance_ref"])
    budget_sec = float(TARGET_METHOD_RUN_SEC)

    logger.info(
        f"Hybrid evaluation on {current_instance_key} | "
        f"K_ref={k_ref}, D_ref={d_ref:.2f}, budget={budget_sec:.2f}s"
    )

    budget_rows: list[dict[str, float | str]] = []
    ttt_k_rows: list[dict[str, float | str]] = []
    ttt_d_rows: list[dict[str, float | str]] = []

    ttt_k_plot_data: dict[str, list[float]] = {}
    ttt_d_plot_data_by_gap: dict[float, dict[str, list[float]]] = {
        float(gap): {} for gap in D_GAPS
    }

    for method_label in method_eval_order:
        if method_label not in method_summaries:
            continue
        summary = method_summaries[method_label]
        run_records = list(summary.get("run_records", []))
        run_count = len(run_records)
        if run_count == 0:
            continue

        final_pairs = [
            (int(record["vehicles"]), float(record["distance"]))
            for record in run_records
        ]

        budget_pairs: list[tuple[int, float]] = []
        ttt_k_times: list[float] = []
        ttt_d_times_by_gap: dict[float, list[float]] = {
            float(gap): [] for gap in D_GAPS
        }

        for record in run_records:
            trace = list(record.get("trace", []))
            if not trace:
                continue

            budget_pair = best_pair_within_budget(trace, budget_sec)
            if budget_pair is not None:
                budget_pairs.append(budget_pair)

            if TTT_TARGETS.get("vehicles", True):
                t_k = first_hit_time(trace, k_target=k_ref)
                ttt_k_times.append(float(t_k))

            if TTT_TARGETS.get("distance", True):
                for gap in D_GAPS:
                    gap_float = float(gap)
                    d_target = (1.0 + gap_float) * d_ref
                    t_d = first_hit_time(trace, d_target=d_target)
                    ttt_d_times_by_gap[gap_float].append(float(t_d))

        avg_final_k = float(np.mean([pair[0] for pair in final_pairs]))
        avg_final_d = float(np.mean([pair[1] for pair in final_pairs]))
        avg_budget_k = (
            float(np.mean([pair[0] for pair in budget_pairs])) if budget_pairs else float("nan")
        )
        avg_budget_d = (
            float(np.mean([pair[1] for pair in budget_pairs])) if budget_pairs else float("nan")
        )
        avg_final_k_gap = float(avg_final_k - float(k_ref))
        avg_budget_k_gap = (
            float(avg_budget_k - float(k_ref)) if np.isfinite(avg_budget_k) else float("nan")
        )

        budget_rows.append(
            {
                "Method": method_label,
                "K Target": k_ref,
                "Target Distance": d_ref,
                "Avg Final K": avg_final_k,
                "Avg Final K Gap": avg_final_k_gap,
                "Avg Final D": avg_final_d,
                f"Avg Budget K @ {budget_sec:.0f}s": avg_budget_k,
                f"Avg Budget K Gap @ {budget_sec:.0f}s": avg_budget_k_gap,
                f"Avg Budget D @ {budget_sec:.0f}s": avg_budget_d,
            }
        )

        if TTT_TARGETS.get("vehicles", True):
            finite_k = finite_values(ttt_k_times)
            ttt_k_rows.append(
                {
                    "Method": method_label,
                    "Runs": run_count,
                    "K Target": k_ref,
                    "Avg Final K Gap": avg_final_k_gap,
                    "K Success Rate": float(len(finite_k) / run_count),
                    "K Median Hit Time (s)": median_or_nan(ttt_k_times),
                }
            )
            ttt_k_plot_data[method_label] = ttt_k_times

        if TTT_TARGETS.get("distance", True):
            for gap in D_GAPS:
                gap_float = float(gap)
                d_target = (1.0 + gap_float) * d_ref
                times_for_gap = ttt_d_times_by_gap[gap_float]
                finite_d = finite_values(times_for_gap)
                ttt_d_rows.append(
                    {
                        "Method": method_label,
                        "Gap (%)": int(gap_float * 100),
                        "Runs": run_count,
                        "Target K": k_ref,
                        "Target Distance": d_target,
                        "Success Rate": float(len(finite_d) / run_count),
                        "Median Hit Time (s)": median_or_nan(times_for_gap),
                    }
                )
                ttt_d_plot_data_by_gap[gap_float][method_label] = times_for_gap

    HYBRID_BUDGET_TABLE = sort_by_method(
        pd.DataFrame(budget_rows),
        method_col="Method",
    )
    HYBRID_TTT_K_TABLE = sort_by_method(
        pd.DataFrame(ttt_k_rows),
        method_col="Method",
    )
    HYBRID_TTT_D_TABLE = sort_by_method(
        pd.DataFrame(ttt_d_rows),
        method_col="Method",
        extra_sort_cols=["Gap (%)"],
    )
    HYBRID_ANALYSIS_INSTANCE_KEY = current_instance_key
    HYBRID_ANALYSIS_BUDGET_SEC = budget_sec

    logger.info("\nHybrid Budget Table")
    logger.info(HYBRID_BUDGET_TABLE.to_string(index=False))

    if not HYBRID_TTT_K_TABLE.empty:
        logger.info("\nHybrid Time-to-Target Table (K)")
        logger.info(HYBRID_TTT_K_TABLE.to_string(index=False))

    if not HYBRID_TTT_D_TABLE.empty:
        logger.info("\nHybrid Time-to-Target Table (D)")
        logger.info(HYBRID_TTT_D_TABLE.to_string(index=False))

    if TTT_TARGETS.get("vehicles", True):
        plot_ttt_ecdf(
            ttt_k_plot_data,
            title=f"Time-to-Target ECDF (K <= {k_ref}) | {current_instance_key}",
            xlabel="CPU Time to Hit K Target (s)",
        )

    if TTT_TARGETS.get("distance", True):
        for gap in D_GAPS:
            gap_float = float(gap)
            plot_ttt_ecdf(
                ttt_d_plot_data_by_gap[gap_float],
                title=(
                    f"Time-to-Target ECDF (D <= (1 + {gap_float:.1f}) * D_ref) "
                    f"| {current_instance_key}"
                ),
                xlabel="CPU Time to Hit D Target (s)",
            )

## Hybrid Batch Runner (Representative Subset)

In [ ]:
from contextlib import contextmanager
from project.Metaheuristics import SimulatedAnnealing, TabuSearch, GeneticAlgorithm
from project.mesa_model import VRPOptimizationModel
from project.mesa_model_rl import VRPOptimizationModelRL
from mesa.agent import AgentSet

BATCH_EXECUTE = ACTIVE_BATCH_EXECUTE
BATCH_VERBOSE = ACTIVE_BATCH_VERBOSE
BATCH_RUNS = N_RUNS
BATCH_SEED_START = SEED_START
BATCH_BUDGET_SEC = TARGET_METHOD_RUN_SEC
BATCH_K_GAPS = K_GAPS
BATCH_DISTANCE_GAPS = D_GAPS
BATCH_INSTANCE_GROUPS = INSTANCE_GROUPS
BATCH_MAX_INSTANCES_PER_GROUP = ACTIVE_BATCH_MAX_INSTANCES_PER_GROUP #?

BATCH_SAVE_CSV = True
BATCH_OUTPUT_DIR = BASE_PATH / "Results"

batch_logger = logging.getLogger("BATCH Runner")
batch_logger.setLevel(logging.INFO)

batch_logger.info(
    f"Batch config | execute={BATCH_EXECUTE} | runs={BATCH_RUNS} | "
    f"budget={BATCH_BUDGET_SEC:.2f}s | max/group={BATCH_MAX_INSTANCES_PER_GROUP}"
)


@contextmanager
def _batch_temp_log_levels(verbose: bool):
    tcc_logger = logging.getLogger("TCC")
    mesa_logger = logging.getLogger("project.mesa_model")
    mesa_rl_logger = logging.getLogger("project.mesa_model_rl")
    old_tcc = tcc_logger.level
    old_mesa = mesa_logger.level
    old_mesa_rl = mesa_rl_logger.level
    if not verbose:
        tcc_logger.setLevel(logging.WARNING)
        mesa_logger.setLevel(logging.WARNING)
        mesa_rl_logger.setLevel(logging.WARNING)
    try:
        yield
    finally:
        tcc_logger.setLevel(old_tcc)
        mesa_logger.setLevel(old_mesa)
        mesa_rl_logger.setLevel(old_mesa_rl)


def _batch_first_hit_time(
    trace: list[dict[str, float | int]],
    *,
    k_target: int | None = None,
    d_target: float | None = None,
) -> float:
    for point in trace:
        k_ok = True if k_target is None else int(point["vehicles"]) <= int(k_target)
        d_ok = True if d_target is None else float(point["distance"]) <= float(d_target)
        if k_ok and d_ok:
            return float(point["elapsed_sec"])
    return float("nan")


def _batch_best_pair_within_budget(
    trace: list[dict[str, float | int]],
    budget_sec: float,
) -> tuple[int, float] | None:
    best_pair: tuple[int, float] | None = None
    for point in trace:
        if float(point["elapsed_sec"]) > float(budget_sec):
            continue
        pair = (int(point["vehicles"]), float(point["distance"]))
        if best_pair is None or pair < best_pair:
            best_pair = pair
    return best_pair


def _batch_finite(values: pd.Series) -> pd.Series:
    numeric_values = pd.to_numeric(values, errors="coerce")
    return numeric_values[np.isfinite(numeric_values)]


def _method_categories(values: pd.Series) -> list[str]:
    observed = [str(v) for v in pd.unique(values.astype(str))]
    ordered = [m for m in METHOD_LABELS if m in observed]
    tail = [m for m in observed if m not in ordered]
    return ordered + tail


def _sort_by_method(
    df: pd.DataFrame,
    method_col: str = "Method",
    extra_sort_cols: list[str] | None = None,
) -> pd.DataFrame:
    if df.empty:
        return df
    result = df.copy()
    categories = _method_categories(result[method_col])
    result["_method_order"] = pd.Categorical(
        result[method_col].astype(str),
        categories=categories,
        ordered=True,
    )
    sort_cols = list(extra_sort_cols or []) + ["_method_order"]
    result = result.sort_values(sort_cols).drop(columns=["_method_order"]).reset_index(drop=True)
    return result


def _normalize_enabled_algorithms(
    enabled_algorithms: tuple[str, ...] | list[str] | None,
) -> list[str]:
    canonical_order = ["SA", "TS", "GA"]
    if enabled_algorithms is None:
        return list(canonical_order)

    cleaned: list[str] = []
    for algo in enabled_algorithms:
        normalized = str(algo).strip().upper()
        if normalized not in canonical_order:
            raise ValueError(f"Unsupported MAS algorithm '{algo}'. Expected one of {canonical_order}.")
        if normalized not in cleaned:
            cleaned.append(normalized)

    selected = [algo for algo in canonical_order if algo in cleaned]
    if not selected:
        raise ValueError("At least one MAS algorithm must be enabled.")
    return selected


def _build_mas_model_for_dataset(
    dataset: Dataset,
    *,
    run_seed: int,
    enabled_algorithms: list[str],
) -> VRPOptimizationModel:
    model = VRPOptimizationModel(
        dataset,
        pool_size=MAS_POOL_SIZE,
        seed=run_seed,
        ga_params=MAS_GA_PARAMS,
        sa_params=MAS_SA_PARAMS,
        ts_params=MAS_TS_PARAMS,
        seed_policy=MAS_SEED_POLICY,
        parallel_agents=PARALLEL_AGENTS,
    )

    agent_by_algo = {
        "SA": model.sa_agent,
        "TS": model.ts_agent,
        "GA": model.ga_agent,
    }
    selected_agents = [agent_by_algo[algo] for algo in enabled_algorithms]
    model.agent_set = AgentSet(selected_agents, random=model.random)
    return model


class VRPOptimizationModelRLSubset(VRPOptimizationModelRL):
    def __init__(
        self,
        dataset: Dataset,
        *,
        enabled_algorithms: tuple[str, ...] | list[str] | None = None,
        **kwargs: object,
    ) -> None:
        super().__init__(dataset, **kwargs)
        self.enabled_algorithms = _normalize_enabled_algorithms(enabled_algorithms)
        agent_by_algo = {
            "SA": self.sa_agent,
            "TS": self.ts_agent,
            "GA": self.ga_agent,
        }
        self.agents_list = [agent_by_algo[algo] for algo in self.enabled_algorithms]
        self.agent_set = AgentSet(self.agents_list, random=self.random)

    def select_action(self, state: tuple[int, int]) -> int:
        available_indices = list(range(len(self.agents_list)))
        if self.random.random() < self.epsilon:
            return self.random.choice(available_indices)

        q_values = self.get_q_values(state)
        max_q = max(q_values[idx] for idx in available_indices)
        best_actions = [idx for idx in available_indices if q_values[idx] == max_q]
        return self.random.choice(best_actions)


def _run_mas_summary_for_dataset(
    dataset: Dataset,
    *,
    runs: int,
    seed_start: int,
    method_label: str = "MAS",
    enabled_algorithms: tuple[str, ...] | list[str] | None = None,
) -> dict[str, object]:
    runtimes: list[float] = []
    vehicles: list[int] = []
    distances: list[float] = []
    run_records: list[dict[str, object]] = []

    best_pair: tuple[int, float] | None = None
    best_route = None
    best_history_t: list[float] = []
    best_history_k: list[int] = []
    best_history_d: list[float] = []
    enabled_algorithms_list = _normalize_enabled_algorithms(enabled_algorithms)

    for run_idx in range(runs):
        run_seed = seed_start + run_idx
        model = _build_mas_model_for_dataset(
            dataset,
            run_seed=run_seed,
            enabled_algorithms=enabled_algorithms_list,
        )

        start_time = time.process_time()
        run_trace: list[dict[str, float | int]] = []

        for _ in range(MAS_N_STEPS):
            model.step()
            best_step = model.elite_pool.best()
            if best_step is None:
                continue
            k_step, d_step = best_step.cost_function()
            run_trace.append(
                {
                    "elapsed_sec": float(time.process_time() - start_time),
                    "vehicles": int(k_step),
                    "distance": float(d_step),
                }
            )

        elapsed = float(time.process_time() - start_time)
        best_run = model.elite_pool.best()
        if best_run is None:
            continue

        k_run, d_run = best_run.cost_function()
        runtimes.append(elapsed)
        vehicles.append(int(k_run))
        distances.append(float(d_run))

        if (not run_trace) or run_trace[-1]["elapsed_sec"] < elapsed:
            run_trace.append(
                {
                    "elapsed_sec": elapsed,
                    "vehicles": int(k_run),
                    "distance": float(d_run),
                }
            )

        run_record = {
            "run": int(run_idx + 1),
            "seed": int(run_seed),
            "runtime_sec": float(elapsed),
            "vehicles": int(k_run),
            "distance": float(d_run),
            "trace": run_trace,
        }
        run_records.append(run_record)

        if best_pair is None or (k_run, d_run) < best_pair:
            best_pair = (int(k_run), float(d_run))
            best_route = best_run
            best_history_t = [float(p["elapsed_sec"]) for p in run_trace]
            best_history_k = [int(p["vehicles"]) for p in run_trace]
            best_history_d = [float(p["distance"]) for p in run_trace]

    return {
        "method_label": method_label,
        "dataset_key": f"{dataset.name.lower()}.txt",
        "enabled_algorithms": list(enabled_algorithms_list),
        "runs": int(runs),
        "avg_runtime_sec": float(np.mean(runtimes)) if runtimes else float("nan"),
        "std_runtime_sec": float(np.std(runtimes, ddof=1)) if len(runtimes) > 1 else 0.0,
        "avg_vehicles": float(np.mean(vehicles)) if vehicles else float("nan"),
        "std_vehicles": float(np.std(vehicles, ddof=1)) if len(vehicles) > 1 else 0.0,
        "avg_distance": float(np.mean(distances)) if distances else float("nan"),
        "std_distance": float(np.std(distances, ddof=1)) if len(distances) > 1 else 0.0,
        "best_vehicles": int(best_pair[0]) if best_pair is not None else None,
        "best_distance": float(best_pair[1]) if best_pair is not None else None,
        "best_route": best_route,
        "best_history_t": best_history_t,
        "best_history_k": best_history_k,
        "best_history_d": best_history_d,
        "run_records": run_records,
    }


def _clone_q_table(
    q_table: dict[tuple[int, int], list[float]] | None,
) -> dict[tuple[int, int], list[float]]:
    if not q_table:
        return {}
    return {
        (int(state[0]), int(state[1])): [float(v) for v in values]
        for state, values in q_table.items()
    }


def _clone_action_count_table(
    count_table: dict[tuple[int, int], list[int]] | None,
) -> dict[tuple[int, int], list[int]]:
    if not count_table:
        return {}
    return {
        (int(state[0]), int(state[1])): [int(v) for v in values]
        for state, values in count_table.items()
    }


def _run_mas_rl_summary_for_dataset(
    dataset: Dataset,
    *,
    runs: int,
    seed_start: int,
    method_label: str = "MAS_RL",
    enabled_algorithms: tuple[str, ...] | list[str] | None = None,
    initial_q_table: dict[tuple[int, int], list[float]] | None = None,
    initial_action_count_table: dict[tuple[int, int], list[int]] | None = None,
) -> dict[str, object]:
    runtimes: list[float] = []
    vehicles: list[int] = []
    distances: list[float] = []
    q_states_list: list[int] = []
    visited_states_list: list[int] = []
    run_records: list[dict[str, object]] = []

    best_pair: tuple[int, float] | None = None
    best_route = None
    best_model = None
    best_history_t: list[float] = []
    best_history_k: list[int] = []
    best_history_d: list[float] = []
    enabled_algorithms_list = _normalize_enabled_algorithms(enabled_algorithms)

    for run_idx in range(runs):
        run_seed = seed_start + run_idx
        model_rl = VRPOptimizationModelRLSubset(
            dataset,
            enabled_algorithms=enabled_algorithms_list,
            pool_size=MAS_POOL_SIZE,
            seed=run_seed,
            ga_params=MAS_GA_PARAMS,
            sa_params=MAS_SA_PARAMS,
            ts_params=MAS_TS_PARAMS,
            seed_policy=MAS_SEED_POLICY,
            parallel_agents=PARALLEL_AGENTS,
            initial_q_table=initial_q_table,
            initial_action_count_table=initial_action_count_table,
            **MAS_RL_PARAMS,
        )

        start_time = time.process_time()
        run_trace: list[dict[str, float | int]] = []

        for _ in range(MAS_N_STEPS):
            model_rl.step()
            best_step = model_rl.elite_pool.best()
            if best_step is None:
                continue
            k_step, d_step = best_step.cost_function()
            run_trace.append(
                {
                    "elapsed_sec": float(time.process_time() - start_time),
                    "vehicles": int(k_step),
                    "distance": float(d_step),
                }
            )

        elapsed = float(time.process_time() - start_time)
        best_run = model_rl.elite_pool.best()
        if best_run is None:
            continue

        k_run, d_run = best_run.cost_function()
        q_states = int(len(model_rl.q_table))
        visited_states = int(len(model_rl.action_count_table))


        runtimes.append(elapsed)
        vehicles.append(int(k_run))
        distances.append(float(d_run))
        q_states_list.append(q_states)
        visited_states_list.append(visited_states)

        if (not run_trace) or run_trace[-1]["elapsed_sec"] < elapsed:
            run_trace.append(
                {
                    "elapsed_sec": elapsed,
                    "vehicles": int(k_run),
                    "distance": float(d_run),
                }
            )

        run_record = {
            "run": int(run_idx + 1),
            "seed": int(run_seed),
            "runtime_sec": float(elapsed),
            "vehicles": int(k_run),
            "distance": float(d_run),
            "q_states": int(q_states),
            "visited_states": int(visited_states),
            "trace": run_trace,
        }
        run_records.append(run_record)

        if best_pair is None or (k_run, d_run) < best_pair:
            best_pair = (int(k_run), float(d_run))
            best_route = best_run
            best_model = model_rl
            best_history_t = [float(p["elapsed_sec"]) for p in run_trace]
            best_history_k = [int(p["vehicles"]) for p in run_trace]
            best_history_d = [float(p["distance"]) for p in run_trace]

    return {
        "method_label": method_label,
        "dataset_key": f"{dataset.name.lower()}.txt",
        "enabled_algorithms": list(enabled_algorithms_list),
        "runs": int(runs),
        "avg_runtime_sec": float(np.mean(runtimes)) if runtimes else float("nan"),
        "std_runtime_sec": float(np.std(runtimes, ddof=1)) if len(runtimes) > 1 else 0.0,
        "avg_vehicles": float(np.mean(vehicles)) if vehicles else float("nan"),
        "std_vehicles": float(np.std(vehicles, ddof=1)) if len(vehicles) > 1 else 0.0,
        "avg_distance": float(np.mean(distances)) if distances else float("nan"),
        "std_distance": float(np.std(distances, ddof=1)) if len(distances) > 1 else 0.0,
        "avg_q_states": float(np.mean(q_states_list)) if q_states_list else float("nan"),
        "avg_visited_states": float(np.mean(visited_states_list)) if visited_states_list else float("nan"),
        "best_vehicles": int(best_pair[0]) if best_pair is not None else None,
        "best_distance": float(best_pair[1]) if best_pair is not None else None,
        "best_route": best_route,
        "best_model": best_model,
        "best_q_table": None if best_model is None else _clone_q_table(best_model.q_table),
        "best_action_count_table": None if best_model is None else _clone_action_count_table(best_model.action_count_table),
        "best_history_t": best_history_t,
        "best_history_k": best_history_k,
        "best_history_d": best_history_d,
        "run_records": run_records,
    }


def _run_method_with_progress(
    *,
    method_label: str,
    step_idx: int,
    step_total: int,
    run_callable,
    progress_logger: logging.Logger | None,
    scope_label: str,
) -> dict[str, object]:
    if progress_logger is not None:
        progress_logger.info(
            f"[Batch][{scope_label}] ({step_idx}/{step_total}) {method_label} | starting"
        )

    method_start = time.process_time()
    summary = run_callable()
    method_elapsed = float(time.process_time() - method_start)

    if progress_logger is not None:
        best_k = summary.get("best_vehicles", None)
        best_d = summary.get("best_distance", None)
        if best_k is None or best_d is None:
            progress_logger.info(
                f"[Batch][{scope_label}] ({step_idx}/{step_total}) {method_label} | "
                f"finished in {method_elapsed:.2f}s | best unavailable"
            )
        else:
            progress_logger.info(
                f"[Batch][{scope_label}] ({step_idx}/{step_total}) {method_label} | "
                f"finished in {method_elapsed:.2f}s | best K={int(best_k)}, D={float(best_d):.2f}"
            )

    return summary


def _run_all_methods_for_dataset(
    dataset: Dataset,
    *,
    runs: int,
    seed_start: int,
    progress_logger: logging.Logger | None = None,
    progress_prefix: str = "",
) -> dict[str, dict[str, object]]:
    summaries: dict[str, dict[str, object]] = {}
    scope_label = progress_prefix or dataset.name
    total_methods = 10

    summaries["TS"] = _run_method_with_progress(
        method_label="TS",
        step_idx=1,
        step_total=total_methods,
        run_callable=lambda: MetaheuristicRunner(
            TabuSearch,
            dataset,
            runs=runs,
            seed_start=seed_start,
            **TS_PARAMS,
        ).run(plot_best=False),
        progress_logger=progress_logger,
        scope_label=scope_label,
    )

    summaries["SA"] = _run_method_with_progress(
        method_label="SA",
        step_idx=2,
        step_total=total_methods,
        run_callable=lambda: MetaheuristicRunner(
            SimulatedAnnealing,
            dataset,
            runs=runs,
            seed_start=seed_start,
            **SA_PARAMS,
        ).run(plot_best=False),
        progress_logger=progress_logger,
        scope_label=scope_label,
    )

    summaries["GA"] = _run_method_with_progress(
        method_label="GA",
        step_idx=3,
        step_total=total_methods,
        run_callable=lambda: MetaheuristicRunner(
            GeneticAlgorithm,
            dataset,
            runs=runs,
            seed_start=seed_start,
            **GA_PARAMS,
        ).run(plot_best=False),
        progress_logger=progress_logger,
        scope_label=scope_label,
    )

    summaries["MAS"] = _run_method_with_progress(
        method_label="MAS",
        step_idx=4,
        step_total=total_methods,
        run_callable=lambda: _run_mas_summary_for_dataset(
            dataset,
            runs=runs,
            seed_start=seed_start,
            method_label="MAS",
        ),
        progress_logger=progress_logger,
        scope_label=scope_label,
    )

    summaries["MAS RL"] = _run_method_with_progress(
        method_label="MAS RL",
        step_idx=5,
        step_total=total_methods,
        run_callable=lambda: _run_mas_rl_summary_for_dataset(
            dataset,
            runs=runs,
            seed_start=seed_start,
            method_label="MAS RL",
        ),
        progress_logger=progress_logger,
        scope_label=scope_label,
    )

    summaries["MAS SA"] = _run_method_with_progress(
        method_label="MAS SA",
        step_idx=6,
        step_total=total_methods,
        run_callable=lambda: _run_mas_summary_for_dataset(
            dataset,
            runs=runs,
            seed_start=seed_start,
            method_label="MAS SA",
            enabled_algorithms=["SA"],
        ),
        progress_logger=progress_logger,
        scope_label=scope_label,
    )

    summaries["MAS TS"] = _run_method_with_progress(
        method_label="MAS TS",
        step_idx=7,
        step_total=total_methods,
        run_callable=lambda: _run_mas_summary_for_dataset(
            dataset,
            runs=runs,
            seed_start=seed_start,
            method_label="MAS TS",
            enabled_algorithms=["TS"],
        ),
        progress_logger=progress_logger,
        scope_label=scope_label,
    )

    summaries["MAS GA"] = _run_method_with_progress(
        method_label="MAS GA",
        step_idx=8,
        step_total=total_methods,
        run_callable=lambda: _run_mas_summary_for_dataset(
            dataset,
            runs=runs,
            seed_start=seed_start,
            method_label="MAS GA",
            enabled_algorithms=["GA"],
        ),
        progress_logger=progress_logger,
        scope_label=scope_label,
    )

    summaries["MAS (TS+SA)"] = _run_method_with_progress(
        method_label="MAS (TS+SA)",
        step_idx=9,
        step_total=total_methods,
        run_callable=lambda: _run_mas_summary_for_dataset(
            dataset,
            runs=runs,
            seed_start=seed_start,
            method_label="MAS (TS+SA)",
            enabled_algorithms=["TS", "SA"],
        ),
        progress_logger=progress_logger,
        scope_label=scope_label,
    )

    summaries["MAS RL (TS+SA)"] = _run_method_with_progress(
        method_label="MAS RL (TS+SA)",
        step_idx=10,
        step_total=total_methods,
        run_callable=lambda: _run_mas_rl_summary_for_dataset(
            dataset,
            runs=runs,
            seed_start=seed_start,
            method_label="MAS RL (TS+SA)",
            enabled_algorithms=["TS", "SA"],
        ),
        progress_logger=progress_logger,
        scope_label=scope_label,
    )

    return summaries


def _build_instance_summary_table(
    run_level_df: pd.DataFrame,
    k_gaps: list[float],
    d_gaps: list[float],
) -> pd.DataFrame:
    if run_level_df.empty:
        return pd.DataFrame()

    rows: list[dict[str, object]] = []
    grouped = run_level_df.groupby(["group", "instance", "method"], dropna=False)
    for (group_name, instance_name, method_label), method_df in grouped:
        k_target_mean = float(method_df["k_target"].mean())
        row: dict[str, object] = {
            "Group": group_name,
            "Instance": instance_name,
            "Method": method_label,
            "Runs": int(len(method_df)),
            "K Target": int(round(k_target_mean)),
            "Avg Final K": float(method_df["final_k"].mean()),
            "Avg Final K Gap": float(method_df["final_k_gap"].mean()),
            "Avg Final D": float(method_df["final_d"].mean()),
            f"Avg Budget K @ {BATCH_BUDGET_SEC:.0f}s": float(method_df["budget_k"].mean()),
            f"Avg Budget K Gap @ {BATCH_BUDGET_SEC:.0f}s": float(method_df["budget_k_gap"].mean()),
            f"Avg Budget D @ {BATCH_BUDGET_SEC:.0f}s": float(method_df["budget_d"].mean()),
        }

        finite_k = _batch_finite(method_df["k_hit_time_sec"])
        row["K Success Rate"] = float(len(finite_k) / len(method_df))
        row["K Median Hit Time (s)"] = float(np.median(finite_k)) if not finite_k.empty else float("nan")

        for gap in k_gaps:
            gap_pct = int(round(float(gap) * 100))
            k_col = f"k_hit_time_sec_gap_{gap_pct}"
            k_target_col = f"k_target_gap_{gap_pct}"
            row[f"Gap {gap_pct}% Target K"] = float(method_df[k_target_col].mean())
            finite_k_gap = _batch_finite(method_df[k_col])
            row[f"Gap {gap_pct}% K Success Rate"] = float(len(finite_k_gap) / len(method_df))
            row[f"Gap {gap_pct}% K Median Hit Time (s)"] = (
                float(np.median(finite_k_gap)) if not finite_k_gap.empty else float("nan")
            )

        for gap in d_gaps:
            gap_pct = int(round(float(gap) * 100))
            d_col = f"d_hit_time_sec_gap_{gap_pct}"
            d_target_col = f"d_target_gap_{gap_pct}"
            row[f"Gap {gap_pct}% Target Distance"] = float(method_df[d_target_col].mean())
            finite_d = _batch_finite(method_df[d_col])
            row[f"Gap {gap_pct}% Success Rate"] = float(len(finite_d) / len(method_df))
            row[f"Gap {gap_pct}% Median Hit Time (s)"] = (
                float(np.median(finite_d)) if not finite_d.empty else float("nan")
            )
        row["K Median Hit Time (s)"] = float(np.median(finite_k)) if not finite_k.empty else float("nan")
        rows.append(row)

    result = pd.DataFrame(rows)
    result = _sort_by_method(result, method_col="Method", extra_sort_cols=["Group", "Instance"])
    return result


def _build_scope_summary_table(
    run_level_df: pd.DataFrame,
    k_gaps: list[float],
    d_gaps: list[float],
) -> pd.DataFrame:
    if run_level_df.empty:
        return pd.DataFrame()
    rows: list[dict[str, object]] = []

    def append_scope(scope_name: str, scope_df: pd.DataFrame) -> None:
        for method_label, method_df in scope_df.groupby("method", dropna=False):
            k_target_mean = float(method_df["k_target"].mean())
            row: dict[str, object] = {
                "Scope": scope_name,
                "Method": method_label,
                "Runs": int(len(method_df)),
                "K Target Avg": k_target_mean,
                "Avg Final K": float(method_df["final_k"].mean()),
                "Avg Final K Gap": float(method_df["final_k_gap"].mean()),
                "Avg Final D": float(method_df["final_d"].mean()),
                f"Avg Budget K @ {BATCH_BUDGET_SEC:.0f}s": float(method_df["budget_k"].mean()),
                f"Avg Budget K Gap @ {BATCH_BUDGET_SEC:.0f}s": float(method_df["budget_k_gap"].mean()),
                f"Avg Budget D @ {BATCH_BUDGET_SEC:.0f}s": float(method_df["budget_d"].mean()),
            }

            finite_k = _batch_finite(method_df["k_hit_time_sec"])
            row["K Success Rate"] = float(len(finite_k) / len(method_df))
            row["K Median Hit Time (s)"] = float(np.median(finite_k)) if not finite_k.empty else float("nan")

            for gap in k_gaps:
                gap_pct = int(round(float(gap) * 100))
                k_col = f"k_hit_time_sec_gap_{gap_pct}"
                k_target_col = f"k_target_gap_{gap_pct}"
                row[f"Gap {gap_pct}% Target K Avg"] = float(method_df[k_target_col].mean())
                finite_k_gap = _batch_finite(method_df[k_col])
                row[f"Gap {gap_pct}% K Success Rate"] = float(len(finite_k_gap) / len(method_df))
                row[f"Gap {gap_pct}% K Median Hit Time (s)"] = (
                    float(np.median(finite_k_gap)) if not finite_k_gap.empty else float("nan")
                )

            for gap in d_gaps:
                gap_pct = int(round(float(gap) * 100))
                d_col = f"d_hit_time_sec_gap_{gap_pct}"
                d_target_col = f"d_target_gap_{gap_pct}"
                row[f"Gap {gap_pct}% Target Distance Avg"] = float(method_df[d_target_col].mean())
                finite_d = _batch_finite(method_df[d_col])
                row[f"Gap {gap_pct}% Success Rate"] = float(len(finite_d) / len(method_df))
                row[f"Gap {gap_pct}% Median Hit Time (s)"] = (
                    float(np.median(finite_d)) if not finite_d.empty else float("nan")
                )
            row["K Median Hit Time (s)"] = float(np.median(finite_k)) if not finite_k.empty else float("nan")
            rows.append(row)

    for group_name, group_df in run_level_df.groupby("group", dropna=False):
        append_scope(str(group_name), group_df)
    append_scope("ALL", run_level_df)

    result = pd.DataFrame(rows)
    if result.empty:
        return result

    scope_categories = list(BATCH_INSTANCE_GROUPS.keys()) + ["ALL"]
    result["_scope_order"] = pd.Categorical(
        result["Scope"].astype(str),
        categories=scope_categories,
        ordered=True,
    )
    result = _sort_by_method(result, method_col="Method", extra_sort_cols=["_scope_order"])
    return result.drop(columns=["_scope_order"])


selected_groups: dict[str, list[str]] = {}
for group_name, group_instances in BATCH_INSTANCE_GROUPS.items():
    if BATCH_MAX_INSTANCES_PER_GROUP is None:
        selected_groups[group_name] = list(group_instances)
    else:
        selected_groups[group_name] = list(group_instances)[: int(BATCH_MAX_INSTANCES_PER_GROUP)]

batch_scope_instance_count = sum(len(group_instances) for group_instances in selected_groups.values())
batch_scope_method_runs = METHOD_COUNT * BATCH_RUNS * batch_scope_instance_count
batch_scope_target_total_sec = BATCH_BUDGET_SEC * batch_scope_method_runs
batch_logger.info(
    f"Hybrid batch target | instances={batch_scope_instance_count} | "
    f"method_runs={batch_scope_method_runs} | target_total={batch_scope_target_total_sec / 60.0:.1f} min"
)

batch_plan_rows: list[dict[str, object]] = []
for group_name, group_instances in selected_groups.items():
    for instance_name in group_instances:
        batch_plan_rows.append(
            {
                "Group": group_name,
                "Instance": instance_name,
                "Has Reference": instance_name in HYBRID_REFERENCE_TABLE,
            }
        )

BATCH_PLAN_TABLE = pd.DataFrame(batch_plan_rows)
batch_logger.info("\nHybrid batch plan")
if BATCH_PLAN_TABLE.empty:
    batch_logger.info("No instances selected for batch execution.")
else:
    batch_logger.info(BATCH_PLAN_TABLE.to_string(index=False))
batch_scope_target_total_sec = BATCH_BUDGET_SEC * batch_scope_method_runs
BATCH_TOTAL_CPU_SEC = float("nan")

if not BATCH_EXECUTE:
    batch_logger.info(
        "Batch runner configured. Set BATCH_EXECUTE = True and rerun this cell to launch all instances."
    )
else:
    BATCH_RUN_LEVEL_ROWS: list[dict[str, object]] = []
    BATCH_SKIPPED_ROWS: list[dict[str, object]] = []
    batch_cpu_start = time.process_time()
    total_groups = len(selected_groups)
    total_instances = int(batch_scope_instance_count)
    completed_instances = 0
    skipped_instances = 0

    batch_logger.info(
        f"[Batch] Execution started | groups={total_groups} | "
        f"instances={total_instances} | methods={METHOD_COUNT} | runs={BATCH_RUNS}"
    )

    with _batch_temp_log_levels(BATCH_VERBOSE):
        for group_idx, (group_name, group_instances) in enumerate(selected_groups.items(), start=1):
            batch_logger.info(
                f"[Batch] Group {group_idx}/{total_groups} | {group_name} | "
                f"instances={len(group_instances)}"
            )

            for instance_idx, instance_name in enumerate(group_instances, start=1):
                global_instance_idx = completed_instances + skipped_instances + 1
                instance_label = f"{group_name}/{instance_name}"
                batch_logger.info(
                    f"[Batch] Instance {global_instance_idx}/{total_instances} | "
                    f"{instance_label} | group_pos={instance_idx}/{len(group_instances)} | starting"
                )

                reference = HYBRID_REFERENCE_TABLE.get(instance_name, None)
                if reference is None:
                    skipped_instances += 1
                    BATCH_SKIPPED_ROWS.append(
                        {
                            "Group": group_name,
                            "Instance": instance_name,
                            "Reason": "missing_reference",
                        }
                    )
                    batch_logger.info(
                        f"[Batch] Instance {global_instance_idx}/{total_instances} | "
                        f"{instance_label} | skipped: missing benchmark reference"
                    )
                    continue

                dataset_path = BASE_PATH / "Datasets" / instance_name
                if not dataset_path.exists():
                    skipped_instances += 1
                    BATCH_SKIPPED_ROWS.append(
                        {
                            "Group": group_name,
                            "Instance": instance_name,
                            "Reason": "missing_dataset_file",
                        }
                    )
                    batch_logger.info(
                        f"[Batch] Instance {global_instance_idx}/{total_instances} | "
                        f"{instance_label} | skipped: missing dataset file"
                    )
                    continue

                instance_cpu_start = time.process_time()
                ds_batch = Dataset(dataset_path)
                method_summaries = _run_all_methods_for_dataset(
                    ds_batch,
                    runs=int(BATCH_RUNS),
                    seed_start=int(BATCH_SEED_START),
                    progress_logger=batch_logger,
                    progress_prefix=instance_label,
                )

                k_ref = int(reference["vehicles_ref"])
                d_ref = float(reference["distance_ref"])
                method_iter_order = [m for m in METHOD_LABELS if m in method_summaries]
                method_iter_order += [
                    m for m in method_summaries.keys() if m not in method_iter_order
                ]

                for method_label in method_iter_order:
                    summary = method_summaries[method_label]
                    run_records = list(summary.get("run_records", []))
                    for record in run_records:
                        trace = list(record.get("trace", []))
                        if not trace:
                            continue

                        budget_pair = _batch_best_pair_within_budget(trace, float(BATCH_BUDGET_SEC))
                        row: dict[str, object] = {
                            "group": group_name,
                            "instance": instance_name,
                            "method": method_label,
                            "run": int(record["run"]),
                            "seed": int(record["seed"]),
                            "runtime_sec": float(record["runtime_sec"]),
                            "final_k": int(record["vehicles"]),
                            "final_k_gap": float(record["vehicles"]) - float(k_ref),
                            "final_d": float(record["distance"]),
                            "budget_k": float("nan") if budget_pair is None else float(budget_pair[0]),
                            "budget_k_gap": (
                                float("nan") if budget_pair is None else float(budget_pair[0]) - float(k_ref)
                            ),
                            "budget_d": float("nan") if budget_pair is None else float(budget_pair[1]),
                            "k_target": int(k_ref),
                            "k_hit_time_sec": _batch_first_hit_time(trace, k_target=k_ref),
                        }

                        for k_gap in BATCH_K_GAPS:
                            gap_pct = int(round(float(k_gap) * 100))
                            k_target = int(np.ceil((1.0 + float(k_gap)) * float(k_ref)))
                            row[f"k_target_gap_{gap_pct}"] = int(k_target)
                            row[f"k_hit_time_sec_gap_{gap_pct}"] = _batch_first_hit_time(
                                trace,
                                k_target=k_target,
                            )

                        for gap in BATCH_DISTANCE_GAPS:
                            gap_pct = int(round(float(gap) * 100))
                            d_target = (1.0 + float(gap)) * d_ref
                            row[f"d_target_gap_{gap_pct}"] = float(d_target)
                            row[f"d_hit_time_sec_gap_{gap_pct}"] = _batch_first_hit_time(
                                trace,
                                d_target=d_target,
                            )

                        BATCH_RUN_LEVEL_ROWS.append(row)
                completed_instances += 1
                instance_elapsed = float(time.process_time() - instance_cpu_start)
                batch_logger.info(
                    f"[Batch] Instance {global_instance_idx}/{total_instances} | "
                    f"{instance_label} | completed in {instance_elapsed:.2f}s"
                )
    batch_logger.info(
        f"[Batch] Execution finished | completed={completed_instances} | skipped={skipped_instances}"
    )

    BATCH_TOTAL_CPU_SEC = float(time.process_time() - batch_cpu_start)
    logger.info(
        f"Hybrid batch measured runtime: {BATCH_TOTAL_CPU_SEC / 60.0:.1f} min "
        f"(target={batch_scope_target_total_sec / 60.0:.1f} min)"
    )

    BATCH_RUN_LEVEL_DF = pd.DataFrame(BATCH_RUN_LEVEL_ROWS)
    BATCH_SKIPPED_DF = pd.DataFrame(BATCH_SKIPPED_ROWS)
    BATCH_INSTANCE_SUMMARY_DF = _build_instance_summary_table(BATCH_RUN_LEVEL_DF, BATCH_K_GAPS, BATCH_DISTANCE_GAPS)
    BATCH_SCOPE_SUMMARY_DF = _build_scope_summary_table(BATCH_RUN_LEVEL_DF, BATCH_K_GAPS, BATCH_DISTANCE_GAPS)

    logger.info("\nHybrid batch run-level rows")
    logger.info(str(len(BATCH_RUN_LEVEL_DF)))

    if not BATCH_INSTANCE_SUMMARY_DF.empty:
        logger.info("\nHybrid batch instance summary")
        logger.info(BATCH_INSTANCE_SUMMARY_DF.to_string(index=False))
    else:
        logger.info("No instance summary generated.")

    if not BATCH_SCOPE_SUMMARY_DF.empty:
        logger.info("\nHybrid batch scope summary")
        logger.info(BATCH_SCOPE_SUMMARY_DF.to_string(index=False))
    else:
        logger.info("No scope summary generated.")

    if not BATCH_SKIPPED_DF.empty:
        logger.info("\nHybrid batch skipped instances")
        logger.info(BATCH_SKIPPED_DF.to_string(index=False))

    if BATCH_SAVE_CSV and not BATCH_RUN_LEVEL_DF.empty:
        BATCH_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
        batch_tag = time.strftime("%Y%m%d_%H%M%S")
        run_level_path = BATCH_OUTPUT_DIR / f"hybrid_batch_run_level_{batch_tag}.csv"
        instance_summary_path = BATCH_OUTPUT_DIR / f"hybrid_batch_instance_summary_{batch_tag}.csv"
        scope_summary_path = BATCH_OUTPUT_DIR / f"hybrid_batch_scope_summary_{batch_tag}.csv"

        BATCH_RUN_LEVEL_DF.to_csv(run_level_path, index=False)
        BATCH_INSTANCE_SUMMARY_DF.to_csv(instance_summary_path, index=False)
        BATCH_SCOPE_SUMMARY_DF.to_csv(scope_summary_path, index=False)

        logger.info("\nHybrid batch CSV files saved:")
        logger.info(str(run_level_path))
        logger.info(str(scope_summary_path))
        logger.info(str(instance_summary_path))


In [ ]:
# Batch TTT ECDF focused view: aggregate all available K targets (overall + per scope)
from pathlib import Path
import json
import logging

BATCH_TTT_PLOT_ENABLED = True
BATCH_TTT_MAX_TIME_SEC = None  # Set a float (e.g., 500.0) to cap x-axis
BATCH_TTT_USE_LATEST_CSV_IF_MISSING = True
BATCH_TTT_K_SHOW_CENSORED = True  # Plot not-hit runs as x markers at runtime_sec
BATCH_TTT_INCLUDE_BASE_K_TARGET = True  # Include exact K target (k_hit_time_sec)
BATCH_TTT_PLOT_ALL_SCOPE = True
BATCH_TTT_PLOT_EACH_SCOPE = True

BATCH_TTT_SAVE_OUTPUTS = True
BATCH_TTT_OUTPUT_ROOT = BASE_PATH / "Results" / "hybrid_batch_ttt"

if "logger" not in globals():
    logging.basicConfig(level=logging.INFO, format="%(message)s")
    logger = logging.getLogger("TCC")


def _batch_ttt_method_order(df: pd.DataFrame) -> list[str]:
    reference_labels = [
        str(v)
        for v in globals().get(
            "METHOD_LABELS",
            ["TS", "SA", "GA", "MAS", "MAS RL", "MAS SA", "MAS TS", "MAS GA", "MAS (TS+SA)", "MAS RL (TS+SA)"],
        )
    ]
    observed = [str(v) for v in pd.unique(df["method"].astype(str))]
    ordered = [m for m in reference_labels if m in observed]
    ordered += [m for m in observed if m not in ordered]
    return ordered


def _batch_latest_run_level_csv() -> Path | None:
    candidate_files: list[Path] = []

    explicit_paths = [
        globals().get("BATCH_LAST_RUN_LEVEL_CSV", None),
        globals().get("BATCH_TTT_SOURCE_CSV_PATH", None),
    ]
    for raw_path in explicit_paths:
        if raw_path is None:
            continue
        path = Path(raw_path)
        if path.exists() and path.is_file():
            candidate_files.append(path.resolve())

    candidate_dirs: list[Path] = []
    if "BATCH_OUTPUT_DIR" in globals() and globals()["BATCH_OUTPUT_DIR"] is not None:
        candidate_dirs.append(Path(globals()["BATCH_OUTPUT_DIR"]))
    candidate_dirs.append(BASE_PATH / "Results" / "hybrid_batch")
    candidate_dirs.append(BASE_PATH / "Results")
    candidate_dirs.append(Path("..") / "Results")
    candidate_dirs.append(Path("Results"))

    seen_dirs: set[Path] = set()
    for raw_dir in candidate_dirs:
        directory = raw_dir.resolve()
        if directory in seen_dirs:
            continue
        seen_dirs.add(directory)
        if not directory.exists() or not directory.is_dir():
            continue

        candidate_files.extend(directory.glob("hybrid_batch_run_level_*.csv"))
        candidate_files.extend(directory.glob("**/run_level.csv"))

    valid_files: list[Path] = []
    seen_files: set[Path] = set()
    for file_path in candidate_files:
        resolved = file_path.resolve()
        if resolved in seen_files:
            continue
        seen_files.add(resolved)
        if resolved.exists() and resolved.is_file():
            valid_files.append(resolved)

    if not valid_files:
        return None

    return max(valid_files, key=lambda p: p.stat().st_mtime)


def _batch_get_run_level_source() -> tuple[pd.DataFrame | None, str, Path | None]:
    if (
        "BATCH_RUN_LEVEL_DF" in globals()
        and isinstance(BATCH_RUN_LEVEL_DF, pd.DataFrame)
        and not BATCH_RUN_LEVEL_DF.empty
    ):
        source_path = None
        raw_path = globals().get("BATCH_LAST_RUN_LEVEL_CSV", None)
        if raw_path is not None:
            path_obj = Path(raw_path)
            if path_obj.exists() and path_obj.is_file():
                source_path = path_obj.resolve()
        return BATCH_RUN_LEVEL_DF.copy(), "memory:BATCH_RUN_LEVEL_DF", source_path

    if not BATCH_TTT_USE_LATEST_CSV_IF_MISSING:
        return None, "missing", None

    latest_csv = _batch_latest_run_level_csv()
    if latest_csv is None:
        return None, "missing", None

    df = pd.read_csv(latest_csv)
    if df.empty:
        return None, f"csv:{latest_csv} (empty)", latest_csv

    globals()["BATCH_TTT_SOURCE_CSV_PATH"] = str(latest_csv)
    return df, f"csv:{latest_csv}", latest_csv


def _batch_detect_gap_columns(df: pd.DataFrame, prefix: str) -> list[tuple[int, str]]:
    pairs: list[tuple[int, str]] = []
    for col in df.columns:
        if not col.startswith(prefix):
            continue
        suffix = col[len(prefix):]
        try:
            gap_pct = int(suffix)
        except ValueError:
            continue
        pairs.append((gap_pct, col))
    pairs.sort(key=lambda item: item[0])
    return pairs


def _batch_build_k_aggregate_frame(
    df: pd.DataFrame,
    *,
    include_base_k: bool = True,
) -> tuple[pd.DataFrame, list[str]]:
    base_cols = ["method"]
    if "runtime_sec" in df.columns:
        base_cols.append("runtime_sec")
    if "group" in df.columns:
        base_cols.append("group")

    frames: list[pd.DataFrame] = []
    labels: list[str] = []

    if include_base_k and "k_hit_time_sec" in df.columns:
        base_frame = df[base_cols + ["k_hit_time_sec"]].copy()
        base_frame["k_target_label"] = "K"
        frames.append(base_frame)
        labels.append("K")

    for gap_pct, col_name in _batch_detect_gap_columns(df, "k_hit_time_sec_gap_"):
        gap_frame = df[base_cols + [col_name]].copy()
        gap_frame = gap_frame.rename(columns={col_name: "k_hit_time_sec"})
        gap_label = f"K+{int(gap_pct)}%"
        gap_frame["k_target_label"] = gap_label
        frames.append(gap_frame)
        labels.append(gap_label)

    if not frames:
        return pd.DataFrame(), []

    return pd.concat(frames, ignore_index=True), labels


def _build_ttt_summary(df: pd.DataFrame, time_col: str) -> pd.DataFrame:
    if "method" not in df.columns or time_col not in df.columns:
        return pd.DataFrame()

    rows: list[dict[str, object]] = []
    scopes: list[tuple[str, pd.DataFrame]] = [("ALL", df)]

    if "group" in df.columns:
        configured_scope_order = [
            str(k) for k in globals().get("BATCH_INSTANCE_GROUPS", {}).keys()
        ]
        observed_scope_order = [str(v) for v in pd.unique(df["group"].astype(str))]
        scope_order = [s for s in configured_scope_order if s in observed_scope_order]
        scope_order += [s for s in observed_scope_order if s not in scope_order]

        for scope_name in scope_order:
            scope_df = df[df["group"].astype(str) == scope_name]
            if scope_df.empty:
                continue
            scopes.append((scope_name, scope_df))

    for scope_name, scope_df in scopes:
        for method_label in _batch_ttt_method_order(scope_df):
            method_df = scope_df[scope_df["method"].astype(str) == method_label]
            total_runs = int(len(method_df))
            if total_runs == 0:
                continue

            values = pd.to_numeric(method_df[time_col], errors="coerce").to_numpy(dtype=float)
            finite_hits = values[np.isfinite(values)]
            success_count = int(len(finite_hits))

            rows.append(
                {
                    "Scope": scope_name,
                    "Method": method_label,
                    "Runs": total_runs,
                    "Success Count": success_count,
                    "Success Rate": float(success_count / total_runs),
                    "Median Hit Time (s)": (
                        float(np.median(finite_hits))
                        if success_count > 0
                        else float("nan")
                    ),
                    "Mean Hit Time (s)": (
                        float(np.mean(finite_hits))
                        if success_count > 0
                        else float("nan")
                    ),
                }
            )

    result = pd.DataFrame(rows)
    if result.empty:
        return result

    method_categories = _batch_ttt_method_order(df)
    scope_categories = list(dict.fromkeys(["ALL"] + [scope for scope, _ in scopes if scope != "ALL"]))
    result["_scope_order"] = pd.Categorical(
        result["Scope"].astype(str),
        categories=scope_categories,
        ordered=True,
    )
    result["_method_order"] = pd.Categorical(
        result["Method"].astype(str),
        categories=method_categories,
        ordered=True,
    )
    result = (
        result
        .sort_values(["_scope_order", "_method_order"])
        .drop(columns=["_scope_order", "_method_order"])
        .reset_index(drop=True)
    )
    return result


def _resolve_ttt_output_dir(source_csv: Path | None) -> Path:
    if "BATCH_LAST_OUTPUT_DIR" in globals():
        output_dir = Path(globals()["BATCH_LAST_OUTPUT_DIR"]) / "ttt"
        return output_dir

    if source_csv is not None and source_csv.name.lower() == "run_level.csv":
        return source_csv.parent / "ttt"

    tag = time.strftime("%Y%m%d_%H%M%S")
    return BATCH_TTT_OUTPUT_ROOT / tag


def _plot_batch_ttt_ecdf(
    df: pd.DataFrame,
    *,
    time_col: str,
    title: str,
    xlabel: str,
    max_time_sec: float | None = None,
    censor_col: str | None = None,
    show_censored: bool = False,
) -> None:
    if "method" not in df.columns or time_col not in df.columns:
        logger.info(f"Skipping plot (missing column): {title}")
        return

    fig, ax = plt.subplots(figsize=(7, 4))
    plotted = False

    for method_label in _batch_ttt_method_order(df):
        method_df = df[df["method"].astype(str) == method_label]
        total_runs = int(len(method_df))
        if total_runs == 0:
            continue

        time_values = pd.to_numeric(method_df[time_col], errors="coerce").to_numpy(dtype=float)
        finite_hits = np.sort(time_values[np.isfinite(time_values)])
        success_count = int(len(finite_hits))
        success_rate = float(success_count / total_runs)
        label = f"{method_label} ({success_count}/{total_runs})"
        line_color = None

        if success_count > 0:
            y = np.arange(1, success_count + 1, dtype=float) / float(total_runs)
            line = ax.step(finite_hits, y, where="post", label=label)[0]
            line_color = line.get_color()
            plotted = True
        elif show_censored:
            line = ax.plot([], [], label=label)[0]
            line_color = line.get_color()

        if show_censored and censor_col is not None and censor_col in method_df.columns:
            censored_mask = ~np.isfinite(time_values)
            if np.any(censored_mask):
                censored_times = pd.to_numeric(
                    method_df.loc[censored_mask, censor_col],
                    errors="coerce",
                ).to_numpy(dtype=float)
                censored_times = censored_times[np.isfinite(censored_times)]
                if max_time_sec is not None:
                    censored_times = np.minimum(censored_times, float(max_time_sec))
                if len(censored_times) > 0:
                    y_cens = np.full(censored_times.shape, success_rate, dtype=float)
                    ax.scatter(
                        censored_times,
                        y_cens,
                        marker="x",
                        s=28,
                        alpha=0.8,
                        color=line_color,
                    )
                    plotted = True

    if not plotted:
        logger.info(f"No successful runs available for plot: {title}")
        plt.close(fig)
        return

    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel("Success Probability")
    ax.set_ylim(0.0, 1.02)
    ax.grid(True, linestyle="--", alpha=0.4)
    ax.legend()
    if max_time_sec is not None:
        ax.set_xlim(0.0, float(max_time_sec))
    fig.tight_layout()
    plt.show()


batch_df, batch_source, batch_source_csv = _batch_get_run_level_source()
if batch_df is None or batch_df.empty:
    logger.info(
        "Batch TTT analysis skipped: no run-level data found in memory and no generated CSV was found."
    )
else:
    logger.info(f"Batch TTT source -> {batch_source}")
    if BATCH_TTT_K_SHOW_CENSORED:
        logger.info("K-target plot uses x markers for runs that did not hit target before runtime end.")

    k_gap_pairs = _batch_detect_gap_columns(batch_df, "k_hit_time_sec_gap_")
    d_gap_pairs = _batch_detect_gap_columns(batch_df, "d_hit_time_sec_gap_")

    BATCH_TTT_K_SUMMARY_BY_GAP = {}
    if "k_hit_time_sec" in batch_df.columns:
        BATCH_TTT_K_SUMMARY_BY_GAP[0] = _build_ttt_summary(batch_df, "k_hit_time_sec")
    BATCH_TTT_K_SUMMARY_BY_GAP.update(
        {
            int(gap_pct): _build_ttt_summary(batch_df, col_name)
            for gap_pct, col_name in k_gap_pairs
        }
    )
    BATCH_TTT_D_SUMMARY_BY_GAP = {
        int(gap_pct): _build_ttt_summary(batch_df, col_name)
        for gap_pct, col_name in d_gap_pairs
    }

    batch_k_plot_df, k_target_labels = _batch_build_k_aggregate_frame(
        batch_df,
        include_base_k=bool(BATCH_TTT_INCLUDE_BASE_K_TARGET),
    )
    BATCH_TTT_K_SUMMARY_AGGREGATED_DF = _build_ttt_summary(batch_k_plot_df, "k_hit_time_sec")
    BATCH_TTT_K_SUMMARY_DF = BATCH_TTT_K_SUMMARY_AGGREGATED_DF.copy()

    if BATCH_TTT_SAVE_OUTPUTS:
        output_dir = _resolve_ttt_output_dir(batch_source_csv)
        output_dir.mkdir(parents=True, exist_ok=True)

        for gap_pct, summary_df in BATCH_TTT_K_SUMMARY_BY_GAP.items():
            summary_df.to_csv(output_dir / f"k_ttt_summary_gap_{int(gap_pct)}.csv", index=False)

        if not BATCH_TTT_K_SUMMARY_AGGREGATED_DF.empty:
            BATCH_TTT_K_SUMMARY_AGGREGATED_DF.to_csv(
                output_dir / "k_ttt_summary_all_targets.csv",
                index=False,
            )

        for gap_pct, summary_df in BATCH_TTT_D_SUMMARY_BY_GAP.items():
            summary_df.to_csv(output_dir / f"d_ttt_summary_gap_{int(gap_pct)}.csv", index=False)

        metadata = {
            "generated_at": time.strftime("%Y-%m-%d %H:%M:%S"),
            "source": batch_source,
            "source_csv": None if batch_source_csv is None else str(batch_source_csv),
            "k_targets_for_plot": k_target_labels,
            "available_k_gaps": [int(gap) for gap, _ in k_gap_pairs],
            "available_d_gaps": [int(gap) for gap, _ in d_gap_pairs],
        }
        with (output_dir / "metadata.json").open("w", encoding="utf-8") as fp:
            json.dump(metadata, fp, indent=2)

        globals()["BATCH_TTT_OUTPUT_DIR"] = str(output_dir)
        logger.info(f"Batch TTT summaries saved at: {output_dir}")

    if batch_k_plot_df.empty:
        logger.info("K-target plot skipped: no K target columns were found.")
    elif not BATCH_TTT_PLOT_ENABLED:
        logger.info("Batch TTT plotting is disabled.")
    else:
        target_label = "K + all available K gaps"
        logger.info(f"K targets included in plot: {', '.join(k_target_labels)}")
        xlabel = "CPU Time to Hit K Target (s)"
        censor_col = "runtime_sec" if "runtime_sec" in batch_k_plot_df.columns else None
        show_censored = bool(BATCH_TTT_K_SHOW_CENSORED and censor_col is not None)

        if BATCH_TTT_PLOT_ALL_SCOPE:
            _plot_batch_ttt_ecdf(
                batch_k_plot_df,
                time_col="k_hit_time_sec",
                title=f"Batch TTT ECDF ({target_label}) | ALL",
                xlabel=xlabel,
                max_time_sec=BATCH_TTT_MAX_TIME_SEC,
                censor_col=censor_col,
                show_censored=False,
            )

        if BATCH_TTT_PLOT_EACH_SCOPE:
            if "group" not in batch_k_plot_df.columns:
                logger.info("Scope plots skipped: source data has no 'group' column.")
            else:
                configured_scope_order = [
                    str(k) for k in globals().get("BATCH_INSTANCE_GROUPS", {}).keys()
                ]
                observed_scope_order = [str(v) for v in pd.unique(batch_k_plot_df["group"].astype(str))]
                scope_order = [s for s in configured_scope_order if s in observed_scope_order]
                scope_order += [s for s in observed_scope_order if s not in scope_order]

                for scope_name in scope_order:
                    scope_df = batch_k_plot_df[batch_k_plot_df["group"].astype(str) == scope_name]
                    if scope_df.empty:
                        continue
                    _plot_batch_ttt_ecdf(
                        scope_df,
                        time_col="k_hit_time_sec",
                        title=f"Batch TTT ECDF ({target_label}) | {scope_name}",
                        xlabel=xlabel,
                        max_time_sec=BATCH_TTT_MAX_TIME_SEC,
                        censor_col=censor_col,
                        show_censored=show_censored,
                    )

## Detailed Replay from Representative Subset Winners
Executa, para cada método, um replay detalhado do melhor caso encontrado no batch (mesma base de logs e gráficos).

In [ ]:
import json
from pathlib import Path

DETAILED_AFTER_BATCH_EXECUTE = bool(globals().get("RUN_DETAILED_BEST_CASES_AFTER_BATCH", True))
DETAILED_AFTER_BATCH_PLOT = bool(globals().get("DETAILED_REPLAY_PLOT", True))
DETAILED_SAVE_OUTPUTS = bool(globals().get("DETAILED_SAVE_OUTPUTS", True))

DETAILED_OUTPUT_ROOT = BASE_PATH / "Results" / "detailed_replay"
DETAILED_QTABLE_OUTPUT_ROOT = BASE_PATH / "Results" / "q_tables"


def _detailed_sort_by_method(df: pd.DataFrame, method_col: str = "Method") -> pd.DataFrame:
    if df.empty:
        return df

    method_reference = [str(m) for m in globals().get("METHOD_LABELS", [])]
    observed = [str(v) for v in pd.unique(df[method_col].astype(str))]
    categories = [m for m in method_reference if m in observed]
    categories += [m for m in observed if m not in categories]

    result = df.copy()
    result["_method_order"] = pd.Categorical(
        result[method_col].astype(str),
        categories=categories,
        ordered=True,
    )
    result = result.sort_values(["_method_order"]).drop(columns=["_method_order"]).reset_index(drop=True)
    return result


def _select_best_batch_cases(run_level_df: pd.DataFrame) -> pd.DataFrame:
    if run_level_df.empty:
        return pd.DataFrame()

    rows: list[dict[str, object]] = []
    grouped = run_level_df.groupby("method", dropna=False)
    for method_label, method_df in grouped:
        best_row = method_df.sort_values(
            ["final_k", "final_d", "runtime_sec", "group", "instance", "run"]
        ).iloc[0]
        rows.append(
            {
                "Method": str(method_label),
                "Group": str(best_row["group"]),
                "Instance": str(best_row["instance"]),
                "Run": int(best_row["run"]),
                "Seed": int(best_row["seed"]),
                "Final K": int(best_row["final_k"]),
                "Final D": float(best_row["final_d"]),
                "Runtime (s)": float(best_row["runtime_sec"]),
            }
        )

    result = pd.DataFrame(rows)
    if result.empty:
        return result
    return _detailed_sort_by_method(result, method_col="Method")


def _plot_detailed_summary(summary: dict[str, object], title_prefix: str) -> None:
    history_k = list(summary.get("best_history_k", []))
    history_d = list(summary.get("best_history_d", []))
    if (not DETAILED_AFTER_BATCH_PLOT) or (not history_k) or (not history_d):
        return

    x = range(1, len(history_k) + 1)
    fig, ax1 = plt.subplots(figsize=(6, 3))
    ax1.plot(x, history_k, color="tab:blue", label="Vehicles")
    ax1.set_xlabel("Iteration/Step")
    ax1.set_ylabel("Vehicles (K)", color="tab:blue")
    ax1.tick_params(axis="y", labelcolor="tab:blue")

    ax2 = ax1.twinx()
    ax2.plot(x, history_d, color="tab:orange", label="Distance")
    ax2.set_ylabel("Distance (D)", color="tab:orange")
    ax2.tick_params(axis="y", labelcolor="tab:orange")

    plt.title(f"{title_prefix} - Cost History")
    fig.tight_layout()
    plt.show()


def _latest_batch_run_level_csv_for_replay() -> Path | None:
    candidates: list[Path] = []

    explicit = [
        globals().get("BATCH_LAST_RUN_LEVEL_CSV", None),
        globals().get("BATCH_TTT_SOURCE_CSV_PATH", None),
    ]
    for raw_path in explicit:
        if raw_path is None:
            continue
        path = Path(raw_path)
        if path.exists() and path.is_file():
            candidates.append(path.resolve())

    if "_batch_latest_run_level_csv" in globals():
        try:
            helper_path = _batch_latest_run_level_csv()
        except Exception:
            helper_path = None
        if helper_path is not None:
            helper_path = Path(helper_path)
            if helper_path.exists() and helper_path.is_file():
                candidates.append(helper_path.resolve())

    scan_dirs = [
        BASE_PATH / "Results" / "hybrid_batch",
        BASE_PATH / "Results",
        Path("..") / "Results",
        Path("Results"),
    ]
    for raw_dir in scan_dirs:
        directory = raw_dir.resolve()
        if not directory.exists() or not directory.is_dir():
            continue
        candidates.extend(directory.glob("hybrid_batch_run_level_*.csv"))
        candidates.extend(directory.glob("**/run_level.csv"))

    valid: list[Path] = []
    seen: set[Path] = set()
    for file_path in candidates:
        resolved = file_path.resolve()
        if resolved in seen:
            continue
        seen.add(resolved)
        if resolved.exists() and resolved.is_file():
            valid.append(resolved)

    if not valid:
        return None

    return max(valid, key=lambda p: p.stat().st_mtime)


def _load_run_level_for_detailed_replay() -> pd.DataFrame | None:
    if (
        "BATCH_RUN_LEVEL_DF" in globals()
        and isinstance(BATCH_RUN_LEVEL_DF, pd.DataFrame)
        and not BATCH_RUN_LEVEL_DF.empty
    ):
        return BATCH_RUN_LEVEL_DF.copy()

    latest_csv = _latest_batch_run_level_csv_for_replay()
    if latest_csv is None:
        return None

    loaded_df = pd.read_csv(latest_csv)
    if loaded_df.empty:
        return None

    globals()["BATCH_RUN_LEVEL_DF"] = loaded_df.copy()
    globals()["BATCH_LAST_RUN_LEVEL_CSV"] = str(latest_csv)
    logger.info(f"Detailed replay loaded run-level data from file: {latest_csv}")
    return loaded_df


def _save_rl_tables_from_summary(
    summary: dict[str, object],
    *,
    method_label: str,
    instance_name: str,
    replay_seed: int,
) -> Path | None:
    q_table = summary.get("best_q_table", None)
    action_count_table = summary.get("best_action_count_table", None)

    if not isinstance(q_table, dict) or not q_table:
        return None

    q_rows: list[dict[str, object]] = []
    for state, values in q_table.items():
        if not isinstance(state, tuple) or len(state) != 2:
            continue
        padded = [float(v) for v in list(values)[:3]]
        if len(padded) < 3:
            padded.extend([0.0] * (3 - len(padded)))
        q_rows.append(
            {
                "k": int(state[0]),
                "distance_bin": int(state[1]),
                "sa": float(padded[0]),
                "ts": float(padded[1]),
                "ga": float(padded[2]),
            }
        )

    count_rows: list[dict[str, object]] = []
    if isinstance(action_count_table, dict):
        for state, values in action_count_table.items():
            if not isinstance(state, tuple) or len(state) != 2:
                continue
            padded = [int(v) for v in list(values)[:3]]
            if len(padded) < 3:
                padded.extend([0] * (3 - len(padded)))
            count_rows.append(
                {
                    "k": int(state[0]),
                    "distance_bin": int(state[1]),
                    "sa": int(padded[0]),
                    "ts": int(padded[1]),
                    "ga": int(padded[2]),
                }
            )

    if not q_rows:
        return None

    safe_method = str(method_label).lower()
    safe_instance = str(instance_name).lower().replace(".txt", "")
    tag = time.strftime("%Y%m%d_%H%M%S")

    output_dir = (
        DETAILED_QTABLE_OUTPUT_ROOT
        / safe_method
        / safe_instance
        / f"seed_{int(replay_seed)}"
        / tag
    )
    output_dir.mkdir(parents=True, exist_ok=True)

    q_df = pd.DataFrame(q_rows).sort_values(["k", "distance_bin"]).reset_index(drop=True)
    q_df.to_csv(output_dir / "q_table.csv", index=False)

    if count_rows:
        count_df = pd.DataFrame(count_rows).sort_values(["k", "distance_bin"]).reset_index(drop=True)
    else:
        count_df = pd.DataFrame(columns=["k", "distance_bin", "sa", "ts", "ga"])
    count_df.to_csv(output_dir / "action_count_table.csv", index=False)

    metadata = {
        "generated_at": time.strftime("%Y-%m-%d %H:%M:%S"),
        "method": method_label,
        "instance": instance_name,
        "seed": int(replay_seed),
        "rows_q_table": int(len(q_df)),
        "rows_action_count_table": int(len(count_df)),
    }
    with (output_dir / "metadata.json").open("w", encoding="utf-8") as fp:
        json.dump(metadata, fp, indent=2)

    globals()["LATEST_QTABLE_ARTIFACT_DIR"] = str(output_dir)
    return output_dir


run_level_for_replay = _load_run_level_for_detailed_replay()
required_symbols = [
    "MetaheuristicRunner",
    "SimulatedAnnealing",
    "TabuSearch",
    "GeneticAlgorithm",
    "_run_mas_summary_for_dataset",
    "_run_mas_rl_summary_for_dataset",
]
missing_symbols = [name for name in required_symbols if name not in globals()]

if not DETAILED_AFTER_BATCH_EXECUTE:
    logger.info("Detailed replay after batch is disabled.")
elif run_level_for_replay is None or run_level_for_replay.empty:
    logger.info("Detailed replay skipped: no run-level data available in memory or files.")
elif missing_symbols:
    logger.info(
        "Detailed replay skipped: missing definitions in memory -> "
        + ", ".join(missing_symbols)
    )
    logger.info("Run the Metaheuristics and Hybrid Batch Runner cells once to populate these symbols.")
else:
    BATCH_BEST_CASES_TABLE = _select_best_batch_cases(run_level_for_replay)
    if BATCH_BEST_CASES_TABLE.empty:
        logger.info("Detailed replay skipped: no best cases available.")
    else:
        logger.info("\nBest case selected per method (from representative subset)")
        logger.info(BATCH_BEST_CASES_TABLE.to_string(index=False))

        DETAILED_BEST_CASE_SUMMARIES: dict[str, dict[str, object]] = {}
        detailed_rows: list[dict[str, object]] = []

        detailed_output_dir: Path | None = None
        if DETAILED_SAVE_OUTPUTS:
            detailed_tag = time.strftime("%Y%m%d_%H%M%S")
            detailed_output_dir = DETAILED_OUTPUT_ROOT / detailed_tag
            detailed_output_dir.mkdir(parents=True, exist_ok=True)
            BATCH_BEST_CASES_TABLE.to_csv(detailed_output_dir / "best_cases.csv", index=False)
            globals()["DETAILED_LAST_OUTPUT_DIR"] = str(detailed_output_dir)

        for _, case_row in BATCH_BEST_CASES_TABLE.iterrows():
            method_label = str(case_row["Method"])
            instance_name = str(case_row["Instance"]).lower()
            replay_seed = int(case_row["Seed"])

            dataset_path = BASE_PATH / "Datasets" / instance_name
            if not dataset_path.exists():
                logger.info(f"Detailed replay skip | {method_label} | {instance_name} | missing dataset")
                continue

            ds_detail = Dataset(dataset_path)
            logger.info(
                f"\n[Detailed Replay] {method_label} | instance={instance_name} | seed={replay_seed}"
            )

            if method_label == "TS":
                summary = MetaheuristicRunner(
                    TabuSearch,
                    ds_detail,
                    runs=1,
                    seed_start=replay_seed,
                    **TS_PARAMS,
                ).run(plot_best=False)
            elif method_label == "SA":
                summary = MetaheuristicRunner(
                    SimulatedAnnealing,
                    ds_detail,
                    runs=1,
                    seed_start=replay_seed,
                    **SA_PARAMS,
                ).run(plot_best=False)
            elif method_label == "GA":
                summary = MetaheuristicRunner(
                    GeneticAlgorithm,
                    ds_detail,
                    runs=1,
                    seed_start=replay_seed,
                    **GA_PARAMS,
                ).run(plot_best=False)
            elif method_label == "MAS":
                summary = _run_mas_summary_for_dataset(
                    ds_detail,
                    runs=1,
                    seed_start=replay_seed,
                    method_label="MAS",
                )
            elif method_label == "MAS SA":
                summary = _run_mas_summary_for_dataset(
                    ds_detail,
                    runs=1,
                    seed_start=replay_seed,
                    method_label="MAS SA",
                    enabled_algorithms=["SA"],
                )
            elif method_label == "MAS TS":
                summary = _run_mas_summary_for_dataset(
                    ds_detail,
                    runs=1,
                    seed_start=replay_seed,
                    method_label="MAS TS",
                    enabled_algorithms=["TS"],
                )
            elif method_label == "MAS GA":
                summary = _run_mas_summary_for_dataset(
                    ds_detail,
                    runs=1,
                    seed_start=replay_seed,
                    method_label="MAS GA",
                    enabled_algorithms=["GA"],
                )
            elif method_label == "MAS (TS+SA)":
                summary = _run_mas_summary_for_dataset(
                    ds_detail,
                    runs=1,
                    seed_start=replay_seed,
                    method_label="MAS (TS+SA)",
                    enabled_algorithms=["TS", "SA"],
                )
            elif method_label in {"MAS RL", "MAS_RL"}:
                summary = _run_mas_rl_summary_for_dataset(
                    ds_detail,
                    runs=1,
                    seed_start=replay_seed,
                    method_label="MAS RL",
                )
            elif method_label == "MAS RL (TS+SA)":
                summary = _run_mas_rl_summary_for_dataset(
                    ds_detail,
                    runs=1,
                    seed_start=replay_seed,
                    method_label="MAS RL (TS+SA)",
                    enabled_algorithms=["TS", "SA"],
                )
            else:
                logger.info(f"Detailed replay skip | unsupported method: {method_label}")
                continue

            DETAILED_BEST_CASE_SUMMARIES[method_label] = summary
            globals()["DETAILED_LAST_METHOD"] = method_label
            globals()["DETAILED_LAST_INSTANCE"] = instance_name
            _plot_detailed_summary(
                summary,
                title_prefix=f"Detailed {method_label} on {instance_name}",
            )

            detailed_rows.append(
                {
                    "Method": method_label,
                    "Instance": instance_name,
                    "Seed": replay_seed,
                    "Best K": summary.get("best_vehicles", None),
                    "Best D": summary.get("best_distance", None),
                    "Avg Runtime (s)": summary.get("avg_runtime_sec", None),
                    "Avg Q States": summary.get("avg_q_states", None),
                    "Avg Visited States": summary.get("avg_visited_states", None),
                }
            )

            if method_label in {"MAS RL", "MAS_RL", "MAS RL (TS+SA)"}:
                best_model = summary.get("best_model", None)
                if best_model is not None:
                    mas_rl_multi_best_model = best_model
                run_records = list(summary.get("run_records", []))
                if run_records:
                    mas_rl_multi_best_record = run_records[0]

                qtable_dir = _save_rl_tables_from_summary(
                    summary,
                    method_label=method_label,
                    instance_name=instance_name,
                    replay_seed=replay_seed,
                )
                if qtable_dir is not None:
                    logger.info(f"Saved RL tables: {qtable_dir}")

        if DETAILED_SAVE_OUTPUTS and detailed_output_dir is not None:
            detailed_summary_df = pd.DataFrame(detailed_rows)
            detailed_summary_df.to_csv(detailed_output_dir / "replay_summary.csv", index=False)

            metadata = {
                "generated_at": time.strftime("%Y-%m-%d %H:%M:%S"),
                "runs_replayed": int(len(detailed_summary_df)),
                "source_run_level": globals().get("BATCH_LAST_RUN_LEVEL_CSV", None),
            }
            with (detailed_output_dir / "metadata.json").open("w", encoding="utf-8") as fp:
                json.dump(metadata, fp, indent=2)

            logger.info(f"Detailed replay outputs saved at: {detailed_output_dir}")

Q-Table plots

In [ ]:
import json
from pathlib import Path

action_labels = ["SA", "TS", "GA"]
QTABLE_OUTPUT_ROOT = BASE_PATH / "Results" / "q_tables"
QTABLE_LOAD_LATEST_IF_MODEL_MISSING = True
QTABLE_SAVE_FROM_MEMORY = True


def _q_tables_from_model(model) -> tuple[pd.DataFrame, pd.DataFrame]:
    states = sorted(set(model.q_table.keys()) | set(model.action_count_table.keys()))

    q_rows: list[dict[str, object]] = []
    count_rows: list[dict[str, object]] = []
    for state in states:
        q_values = [float(v) for v in model.q_table.get(state, [0.0, 0.0, 0.0])[:3]]
        while len(q_values) < 3:
            q_values.append(0.0)

        c_values = [int(v) for v in model.action_count_table.get(state, [0, 0, 0])[:3]]
        while len(c_values) < 3:
            c_values.append(0)

        q_rows.append(
            {
                "k": int(state[0]),
                "distance_bin": int(state[1]),
                "sa": float(q_values[0]),
                "ts": float(q_values[1]),
                "ga": float(q_values[2]),
            }
        )
        count_rows.append(
            {
                "k": int(state[0]),
                "distance_bin": int(state[1]),
                "sa": int(c_values[0]),
                "ts": int(c_values[1]),
                "ga": int(c_values[2]),
            }
        )

    q_df = pd.DataFrame(q_rows)
    count_df = pd.DataFrame(count_rows)

    if not q_df.empty:
        q_df = q_df.sort_values(["k", "distance_bin"]).reset_index(drop=True)
    if not count_df.empty:
        count_df = count_df.sort_values(["k", "distance_bin"]).reset_index(drop=True)

    return q_df, count_df


def _build_matrices_from_frames(
    q_df: pd.DataFrame,
    count_df: pd.DataFrame,
) -> tuple[list[tuple[int, int]], np.ndarray, np.ndarray]:
    state_set: set[tuple[int, int]] = set()

    if not q_df.empty and {"k", "distance_bin"}.issubset(q_df.columns):
        state_set.update(
            (int(k), int(d_bin))
            for k, d_bin in q_df[["k", "distance_bin"]].itertuples(index=False, name=None)
        )

    if not count_df.empty and {"k", "distance_bin"}.issubset(count_df.columns):
        state_set.update(
            (int(k), int(d_bin))
            for k, d_bin in count_df[["k", "distance_bin"]].itertuples(index=False, name=None)
        )

    states = sorted(state_set)
    if not states:
        return [], np.array([]), np.array([])

    q_lookup: dict[tuple[int, int], list[float]] = {}
    if not q_df.empty:
        for row in q_df.itertuples(index=False):
            q_lookup[(int(row.k), int(row.distance_bin))] = [
                float(row.sa),
                float(row.ts),
                float(row.ga),
            ]

    count_lookup: dict[tuple[int, int], list[int]] = {}
    if not count_df.empty:
        for row in count_df.itertuples(index=False):
            count_lookup[(int(row.k), int(row.distance_bin))] = [
                int(row.sa),
                int(row.ts),
                int(row.ga),
            ]

    q_values = np.array(
        [q_lookup.get(state, [0.0, 0.0, 0.0]) for state in states],
        dtype=float,
    )
    count_values = np.array(
        [count_lookup.get(state, [0, 0, 0]) for state in states],
        dtype=int,
    )

    return states, q_values, count_values


def _save_qtable_artifact(
    q_df: pd.DataFrame,
    count_df: pd.DataFrame,
    metadata: dict[str, object],
) -> Path | None:
    if q_df.empty:
        return None

    method = str(metadata.get("method", "mas_rl")).lower()
    instance = str(metadata.get("instance", "unknown")).lower().replace(".txt", "")
    seed_value = metadata.get("seed", "na")
    tag = time.strftime("%Y%m%d_%H%M%S")

    output_dir = QTABLE_OUTPUT_ROOT / method / instance / f"seed_{seed_value}" / tag
    output_dir.mkdir(parents=True, exist_ok=True)

    q_df.to_csv(output_dir / "q_table.csv", index=False)
    count_df.to_csv(output_dir / "action_count_table.csv", index=False)

    metadata_to_save = dict(metadata)
    metadata_to_save["generated_at"] = time.strftime("%Y-%m-%d %H:%M:%S")
    metadata_to_save["rows_q_table"] = int(len(q_df))
    metadata_to_save["rows_action_count_table"] = int(len(count_df))

    with (output_dir / "metadata.json").open("w", encoding="utf-8") as fp:
        json.dump(metadata_to_save, fp, indent=2)

    globals()["LATEST_QTABLE_ARTIFACT_DIR"] = str(output_dir)
    return output_dir


def _load_latest_qtable_artifact() -> tuple[pd.DataFrame, pd.DataFrame, dict[str, object], Path] | None:
    if not QTABLE_OUTPUT_ROOT.exists() or not QTABLE_OUTPUT_ROOT.is_dir():
        return None

    q_files = list(QTABLE_OUTPUT_ROOT.glob("**/q_table.csv"))
    if not q_files:
        return None

    latest_q = max(q_files, key=lambda p: p.stat().st_mtime)
    artifact_dir = latest_q.parent
    count_path = artifact_dir / "action_count_table.csv"
    metadata_path = artifact_dir / "metadata.json"

    q_df = pd.read_csv(latest_q)
    count_df = (
        pd.read_csv(count_path)
        if count_path.exists() and count_path.is_file()
        else pd.DataFrame(columns=["k", "distance_bin", "sa", "ts", "ga"])
    )

    metadata: dict[str, object] = {}
    if metadata_path.exists() and metadata_path.is_file():
        try:
            with metadata_path.open("r", encoding="utf-8") as fp:
                metadata = json.load(fp)
        except Exception:
            metadata = {}

    return q_df, count_df, metadata, artifact_dir


best_model = globals().get("mas_rl_multi_best_model", None)
best_record = globals().get("mas_rl_multi_best_record", None)

q_df = pd.DataFrame(columns=["k", "distance_bin", "sa", "ts", "ga"])
count_df = pd.DataFrame(columns=["k", "distance_bin", "sa", "ts", "ga"])
best_suffix = ""

if best_model is not None:
    q_df, count_df = _q_tables_from_model(best_model)

    if best_record:
        best_suffix = f" (Run {best_record['run']}, Seed {best_record['seed']})"

    if QTABLE_SAVE_FROM_MEMORY and not q_df.empty:
        metadata = {
            "source": "in_memory_best_model",
            "method": str(globals().get("DETAILED_LAST_METHOD", "MAS_RL")),
            "instance": str(globals().get("DETAILED_LAST_INSTANCE", "unknown")),
            "seed": int(best_record["seed"]) if best_record and "seed" in best_record else "na",
        }
        saved_dir = _save_qtable_artifact(q_df, count_df, metadata)
        if saved_dir is not None:
            logger.info(f"Saved Q-table artifact: {saved_dir}")
elif QTABLE_LOAD_LATEST_IF_MODEL_MISSING:
    loaded = _load_latest_qtable_artifact()
    if loaded is not None:
        q_df, count_df, loaded_meta, loaded_dir = loaded
        meta_method = loaded_meta.get("method", "unknown")
        meta_seed = loaded_meta.get("seed", "na")
        best_suffix = f" (loaded: {meta_method}, seed={meta_seed})"
        logger.info(f"Loaded Q-table artifact: {loaded_dir}")
    else:
        logger.info("No MAS RL model in memory and no saved Q-table artifact found.")
else:
    logger.info("No MAS RL best model available. Run MAS RL sections first.")

states, q_values, count_values = _build_matrices_from_frames(q_df, count_df)

if not states:
    logger.info("No states available for Q-table visualization.")
else:
    fig1, ax1 = plt.subplots(figsize=(8, max(3, len(states) * 0.45)))
    im1 = ax1.imshow(q_values, cmap="viridis", aspect="auto")
    ax1.set_xticks(np.arange(len(action_labels)))
    ax1.set_xticklabels(action_labels)
    ax1.set_yticks(np.arange(len(states)))
    ax1.set_yticklabels([f"K={k}, Dbin={d_bin}" for k, d_bin in states])
    ax1.set_xlabel("Action")
    ax1.set_ylabel("State")
    ax1.set_title(f"Q-Table{best_suffix}")

    q_abs_max = np.max(np.abs(q_values)) if q_values.size else 0.0
    q_threshold = q_abs_max * 0.5
    for i in range(q_values.shape[0]):
        for j in range(q_values.shape[1]):
            value = q_values[i, j]
            text_color = "white" if abs(value) > q_threshold else "black"
            ax1.text(j, i, f"{value:.2f}", ha="center", va="center", color=text_color, fontsize=8)

    fig1.colorbar(im1, ax=ax1, label="Q-value")
    fig1.tight_layout()
    plt.show()

    algo_totals = np.sum(count_values, axis=0).astype(int)
    logger.info(
        f"Best-run algo totals -> SA: {int(algo_totals[0])}, "
        f"TS: {int(algo_totals[1])}, "
        f"GA: {int(algo_totals[2])}"
    )

    fig2, ax2 = plt.subplots(figsize=(8, max(3, len(states) * 0.45)))
    im2 = ax2.imshow(count_values, cmap="Blues", aspect="auto")
    ax2.set_xticks(np.arange(len(action_labels)))
    ax2.set_xticklabels(action_labels)
    ax2.set_yticks(np.arange(len(states)))
    ax2.set_yticklabels([f"K={k}, Dbin={d_bin}" for k, d_bin in states])
    ax2.set_xlabel("Action")
    ax2.set_ylabel("State")
    ax2.set_title(f"Action Run Counts{best_suffix}")

    c_max = np.max(count_values) if count_values.size else 0
    c_threshold = c_max * 0.5
    for i in range(count_values.shape[0]):
        for j in range(count_values.shape[1]):
            count = int(count_values[i, j])
            text_color = "white" if count > c_threshold else "black"
            ax2.text(j, i, f"{count}", ha="center", va="center", color=text_color, fontsize=8)

    fig2.colorbar(im2, ax=ax2, label="Runs")
    fig2.tight_layout()
    plt.show()

# Appendix

Understanding the size of a NP problem.

Estimates the time required to solve a 100-customer TSP instance using exhaustive search (Brute Force).

In [ ]:
import math

num_customers = 100
n_permuted = num_customers - 1 # 1 depot, so we permute (n-1) customers

total_solutions = math.factorial(n_permuted)

solutions_per_ms = 1_000_000_000
solutions_per_sec = solutions_per_ms * 1_000

sec_per_year = 60 * 60 * 24 * 365.25
solutions_per_year = solutions_per_sec * sec_per_year

estimated_years = total_solutions / solutions_per_year

# Contextual comparison: Estimated age of the universe (13.8 billion years)
age_of_universe = 13.8e9
ratio_universe_age = estimated_years / age_of_universe


print(f"\nESTIMATED TIME TO FIND THE OPTIMAL SOLUTION:")
print(f"{estimated_years:.4e} years")

print(f"\nCOMPARISON WITH COSMIC SCALE:")
print(f"This is approximately {ratio_universe_age:.4e} times the age of the universe.")
